# Polymarket Mispricing — ML Pipeline v2

A cleaner, self-contained version of `scripts.ipynb` for the CBS Machine Learning & Deep Learning final project. It preserves the same data files, random seed, stages, and output paths, but splits the large helper cells into smaller blocks that are easier to read and rerun.


## Run Guide

**Normal run:** run cells top-to-bottom. The default path downloads data if needed, runs Scripts 01-05, and skips every expensive optional tuning/promotion/refresh cell.

**Optional tuned LightGBM workflow:** after Scripts 01-05 have produced baseline outputs, enable the gated LightGBM tuning cell in Script 06, then enable the gated promotion cell in the Appendix, then enable the calibration and backtest refresh cells.

All generated files still land under `outputs/data`, `outputs/models`, `outputs/metrics`, `outputs/backtest`, and `outputs/tuning`.


## Setup

Install/import dependencies, define repo-relative paths, set the random seed, and load shared plotting/runtime helpers.


### Setup — Dependencies

Installs the required Python packages into the active kernel if they are missing. Expected runtime is usually a few seconds when dependencies are already present.


In [1]:
"""Setup cell: imports, paths, helpers.

NOTEBOOK_DIR resolves to the directory the notebook was launched from, which is
expected to be the repo root (containing data/ and where outputs/ will be created).
"""
from __future__ import annotations

# --- Auto-install dependencies (idempotent: pip is fast for already-satisfied pkgs) ---
# Bakes the requirements directly into the notebook so the examiner can run this in
# a fresh Jupyter env without first running `pip install -r requirements.txt`.
import sys, subprocess as _sp
_DEPS = [
    "numpy>=1.26", "pandas>=2.2", "pyarrow>=15.0", "scikit-learn>=1.4", "joblib>=1.3",
    "matplotlib>=3.8", "seaborn>=0.13",
    "lightgbm>=4.0", "optuna>=3.5", "shap>=0.44",
]
print("Checking dependencies (pip-installing what is missing)...")
_sp.check_call([sys.executable, "-m", "pip", "install", "--quiet", *_DEPS])
print("Dependencies OK.")


Checking dependencies (pip-installing what is missing)...
Dependencies OK.


In [2]:
# --- Mac M-series threading setup ----------------------------------------
# what: pin every parallel pool to the 10 P-cores on M4 Pro (skip the 4 E-cores).
# why: efficiency cores are ~60% slower per core; including them hurts throughput
#      for barrier-heavy workloads (lightgbm histogram phases, HGBM splits).
# Must run BEFORE numpy / sklearn / lightgbm are imported in this process —
# after import the env vars are ignored. Restart the kernel if these libs were
# already loaded by a previous run.
import os

N_JOBS = 10  # P-cores on Apple M4 Pro (10P + 4E = 14 logical CPUs)

for _var in (
    "OMP_NUM_THREADS",          # OpenMP (HistGradientBoosting, lightgbm core)
    "OPENBLAS_NUM_THREADS",     # OpenBLAS (numpy/scipy on this conda env)
    "MKL_NUM_THREADS",          # Intel MKL (no-op on Apple silicon, set anyway)
    "BLIS_NUM_THREADS",         # BLIS
    "NUMEXPR_NUM_THREADS",      # numexpr
    "VECLIB_MAXIMUM_THREADS",   # macOS Accelerate
    "LOKY_MAX_CPU_COUNT",       # joblib's loky backend (sklearn n_jobs=-1)
):
    os.environ.setdefault(_var, str(N_JOBS))

print(f"Threading configured: N_JOBS={N_JOBS}, OMP/BLAS pinned to {N_JOBS} P-cores.")


Threading configured: N_JOBS=10, OMP/BLAS pinned to 10 P-cores.


### Setup — Imports

Imports the standard library, plotting stack, NumPy/Pandas, and scikit-learn objects used by later stages.


In [3]:
# Standard library + scientific stack imports used across all cells.
import argparse
import ast
import json
import os
import shutil
import sys
import time
import warnings
from contextlib import contextmanager
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    IsolationForest,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")


### Setup — Paths And Reproducibility

Defines notebook-relative input/output paths plus the project seed and cross-validation fold count.


In [4]:
# Notebook-relative paths so the repo is portable.
# Launch jupyter from the repo root; data/ must sit alongside this notebook.
NOTEBOOK_DIR = Path.cwd().resolve()

# --- inlined: config.py ---------------------------------------------------
from pathlib import Path

# what: anchor every path off this file's location -> always correct, never a relative-path bug
# why: scripts can be run from anywhere (IDE, terminal, notebook) and still find data
# how: config.py lives in submission/scripts/, so the submission root is one level up
_SUBMISSION_ROOT = NOTEBOOK_DIR

# what: data folder bundled with the submission (consolidated_modeling_data.parquet lives here)
DATA_DIR = _SUBMISSION_ROOT / "data"

# what: where every script writes its outputs (created on demand, gitignored upstream)
OUTPUTS_DIR = _SUBMISSION_ROOT / "outputs"

# what: numerical-reproducibility seed used by every model factory and every np.random call
RANDOM_SEED = 42

# what: number of cross-validation folds used by 03_train_models, 04_calibration, 06_tuning
N_FOLDS = 5


### Setup — Plot Theme

Centralizes figure colors, sizes, and axis cleanup helpers so later plots have one consistent style.


In [5]:
# --- inlined: _design.py --------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Sequential rocket cmap for heatmaps and rank-gradient bars.
C_MAP = sns.color_palette("rocket_r", as_cmap=True)
# Polar-contrast cmap for plots whose visual story is divergence around a
# centre (cool blue, dark centre, warm red).
C_MAP_CONTRAST = sns.color_palette("icefire", as_cmap=True)
# Performance cmap (red = bad, green = good). Reserved for signed maps
# where the direction carries a value judgement (ROI, PnL, calibrated
# residual edge). All other heatmaps use C_MAP.
C_MAP_PERFORMANCE = LinearSegmentedColormap.from_list(
    "perf_rg",
    [(0.0, "#7a1717"), (0.4, "#d96a6a"), (0.5, "#f0f0f0"),
     (0.6, "#7fcf86"), (1.0, "#15703a")],
)
COL_PERF_BAD = "#7a1717"
COL_PERF_GOOD = "#15703a"

# Discrete rocket palette: index 0 (near-black) → 9 (cream).
PAL_10 = sns.color_palette("rocket", 10)
# Seven-colour categorical palette, skips near-black bottom and pale top.
PAL_GROUPS = [PAL_10[i] for i in range(1, 8)]

COL_DARK = "0.15"
# Two-class anchor pair: cool wine-red vs warm red-orange.
COL_TRAIN = COL_CORRECT = COL_BAR = PAL_10[4]
COL_TEST = COL_INCORRECT = COL_BAR_ALT = PAL_10[6]

FIG_W = 6.3
FIG_W_HALF = 3.1
FIG_W_WIDE = 7.8


def rocket_gradient(n: int, lo: float = 0.2, hi: float = 0.95) -> list:
    """Rank-gradient sampled from C_MAP, lightest (low rank) → darkest."""
    if n <= 1:
        return [C_MAP(0.5)]
    return [C_MAP(lo + (hi - lo) * i / (n - 1)) for i in range(n)]


def apply_theme() -> None:
    """Apply the report-style seaborn / matplotlib theme. Idempotent."""
    sns.set_theme(style="white", context="paper")
    plt.rcParams.update({
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "legend.fontsize": 8,
    })


def clean_ax(ax) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


### Setup — Runtime Helpers

Shared utilities for BLAS thread caps, derived seeds, VM paths, wall-clock logging, and oversubscription checks.


In [6]:
import json
import os
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Any

# Default cap inside workers. Override in launcher scripts that need more
# (e.g. 03_train_models_vm uses 32 BLAS threads per outer worker).
DEFAULT_BLAS_PER_WORKER = 1


def cap_blas_threads(n: int = DEFAULT_BLAS_PER_WORKER) -> None:
    """Cap every common BLAS / OpenMP thread pool to n threads.

    Must be called before numpy / sklearn / lightgbm imports in this
    process. After those libs load, the env vars are ignored.
    """
    val = str(int(n))
    # what: every BLAS / OpenMP variant we might encounter
    # why: a single one being unset is enough to spawn 256 helper threads
    #      per worker on a 256-core box; that crushes the scheduler
    for var in (
        "OMP_NUM_THREADS",  # OpenMP (HistGBM, lightgbm core)
        "MKL_NUM_THREADS",  # Intel MKL
        "OPENBLAS_NUM_THREADS",  # OpenBLAS
        "BLIS_NUM_THREADS",  # BLIS
        "NUMEXPR_NUM_THREADS",  # numexpr
        "VECLIB_MAXIMUM_THREADS",  # macOS Accelerate (laptop only)
    ):
        os.environ[var] = val


def worker_init(blas_threads: int = DEFAULT_BLAS_PER_WORKER) -> None:
    """joblib `initializer=` callback. Runs once per worker, before any task."""
    cap_blas_threads(blas_threads)
    # cheap sanity check that the env vars actually stuck
    assert os.environ.get("OMP_NUM_THREADS") == str(blas_threads), (
        f"BLAS cap not set in worker (OMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS')})"
    )


def derive_seed(base_seed: int, *idx: int) -> int:
    """Deterministic 32-bit seed from (base_seed, idx_tuple).

    Use this everywhere a worker needs randomness. Workers run in
    nondeterministic order, so seeding from job index (not from a
    shared global rng) is the only way to keep results reproducible.
    """
    import numpy as np  # local import: keep module top numpy-free

    ss = np.random.SeedSequence(base_seed)
    if not idx:
        return int(ss.generate_state(1)[0])
    # spawn one child per index level; final state is deterministic
    children = ss.spawn(1)
    for i in idx:
        children = children[0].spawn(int(i) + 1)
    return int(children[-1].generate_state(1)[0])


def submission_root() -> Path:
    """Return the submission/ root, derived from this file's location."""
    return NOTEBOOK_DIR


def vm_paths() -> tuple[Path, Path]:
    """(DATA_DIR, OUTPUTS_VM_DIR). Mirrors config.py but with _vm suffix.

    OUTPUTS_VM_DIR is created if missing so VM runs never collide with
    the laptop reference outputs/.
    """
    sub = submission_root()
    data = sub / "data"
    out_vm = sub / "outputs_vm"
    out_vm.mkdir(parents=True, exist_ok=True)
    return data, out_vm


@contextmanager
def wall_clock_log(stage: str, log_path: Path | None = None):
    """Print and (optionally) record wall-clock seconds for a pipeline stage."""
    t0 = time.time()
    print(f"[{stage}] start")
    try:
        yield
    finally:
        elapsed = time.time() - t0
        print(f"[{stage}] done in {elapsed:.1f}s")
        if log_path is not None:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            entries: dict[str, Any] = {}
            if log_path.exists():
                try:
                    entries = json.loads(log_path.read_text())
                except json.JSONDecodeError:
                    pass
            entries[stage] = round(elapsed, 1)
            log_path.write_text(json.dumps(entries, indent=2))


def detect_oversubscription(threshold_load_factor: float = 1.5) -> dict:
    """Heuristic load-average check. Returns warn flag, doesn't raise."""
    try:
        import psutil  # noqa: PLC0415
    except ImportError:
        return {"warn": False, "reason": "psutil not installed"}
    cpus = psutil.cpu_count(logical=False) or 1
    load1 = os.getloadavg()[0] if hasattr(os, "getloadavg") else 0.0
    return {
        "physical_cpus": cpus,
        "load_avg_1min": round(load1, 1),
        "warn": bool(load1 > threshold_load_factor * cpus),
    }


def assert_close(serial: dict, vm: dict, tol: dict[str, float]) -> list[str]:
    """Compare flat metric dicts. Returns list of failed keys (empty = pass).

    Used by tests/test_vm_parity.py to enforce the parity contract:
        AUC tol 1e-3, ROI tol 5e-3, paired-bootstrap CI tol 2e-2.
    """
    failures: list[str] = []
    for key, allowed in tol.items():
        sv = serial.get(key)
        vv = vm.get(key)
        if sv is None or vv is None:
            failures.append(f"{key}: missing (serial={sv}, vm={vv})")
            continue
        diff = abs(float(sv) - float(vv))
        if diff > allowed:
            failures.append(f"{key}: |{sv} - {vv}| = {diff:.4f} > tol {allowed}")
    return failures


def n_workers_default(unit: str = "cell") -> int:
    """Default n_jobs based on the parallel unit and machine size.

    'cell':  fine-grained, no inner BLAS  -> physical_cpus // 4 (cap 64)
    'model': coarse, 32 BLAS per worker  -> 8
    """
    try:
        import psutil  # noqa: PLC0415

        cpus = psutil.cpu_count(logical=False) or 8
    except ImportError:
        cpus = (os.cpu_count() or 8) // 2
    if unit == "cell":
        return min(64, max(1, cpus // 4))
    if unit == "model":
        return min(8, max(1, cpus // 32))
    raise ValueError(f"unknown unit: {unit!r}")


### Setup — Confirm Configuration

Applies the plotting theme and prints the active directories and seed.


In [7]:
apply_theme()
print(f"Setup OK.")
print(f"  NOTEBOOK_DIR = {NOTEBOOK_DIR}")
print(f"  DATA_DIR     = {DATA_DIR}  (exists={DATA_DIR.exists()})")
print(f"  OUTPUTS_DIR  = {OUTPUTS_DIR}")
print(f"  SEED={RANDOM_SEED}  N_FOLDS={N_FOLDS}")
if not (DATA_DIR / "consolidated_modeling_data.parquet").exists():
    print()
    print("WARNING: consolidated_modeling_data.parquet not found in data/.")
    print("Download it from the GitHub release before running Script 01.")
    print("See README.md > 'Get the data'.")


Setup OK.
  NOTEBOOK_DIR = /Users/alex/Documents/GitHub/ML-final
  DATA_DIR     = /Users/alex/Documents/GitHub/ML-final/data  (exists=True)
  OUTPUTS_DIR  = /Users/alex/Documents/GitHub/ML-final/outputs
  SEED=42  N_FOLDS=5


## Fetch Data

Downloads the two parquet files from the GitHub release only when they are missing, then verifies SHA-256 checksums. Skips quickly when files already exist.


In [8]:
import hashlib
import urllib.request

DATA_FILES = [
    {
        "name": "consolidated_modeling_data.parquet",
        "url": "https://github.com/alexandermyrup/ML-final/releases/download/data-v1/consolidated_modeling_data.parquet",
        "sha256": "85f424975f8384590d8cd781d9353e2977a6cf336fa4bda01b1310f65e98cc87",
    },
    {
        "name": "backtest_context.parquet",
        "url": "https://github.com/alexandermyrup/ML-final/releases/download/data-v1/backtest_context.parquet",
        "sha256": "2ccb700cf74f8498c0647840ec25697232e612fec94168b281755f54877ea0b7",
    },
]

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_DIR.mkdir(parents=True, exist_ok=True)
for f in DATA_FILES:
    dst = DATA_DIR / f["name"]
    if dst.exists():
        # Quick integrity check; re-download only if hash mismatches
        if _sha256(dst) == f["sha256"]:
            print(f"  {f['name']}: present and verified ({dst.stat().st_size / 1e6:.1f} MB)")
            continue
        print(f"  {f['name']}: present but checksum mismatch, re-downloading")
        dst.unlink()
    print(f"  {f['name']}: downloading from {f['url']}")
    urllib.request.urlretrieve(f["url"], dst)
    got = _sha256(dst)
    if got != f["sha256"]:
        raise SystemExit(f"checksum mismatch after download: {got} != {f['sha256']}")
    print(f"    -> {dst.stat().st_size / 1e6:.1f} MB, SHA-256 OK")
print("Data ready.")


  consolidated_modeling_data.parquet: present and verified (317.5 MB)
  backtest_context.parquet: present and verified (21.2 MB)
Data ready.


## Script 01 — Data Prep And Leakage Checks

Loads the modeling parquet, creates the train/test split, runs leakage/sanity checks, and writes `outputs/data/feature_cols.json` plus `outputs/data/leakage_report.json`. Runtime is about 30 seconds.


### Script 01 — Constants

Defines the target, metadata columns, forbidden leaky columns, expected row counts, and event timestamps used by the checks.


In [9]:
import ast
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# what: import central paths/seeds so every script in the pipeline reads from one place
# how: config.py lives next to this script
# why: teacher only edits one file if their data path differs
# what: meta columns that are NOT features (book-keeping for the row)
# how: we exclude them when building X for modeling
META_COLS = {"split", "market_id", "ts_dt", "timestamp"}
TARGET = "bet_correct"

# what: modelling scripts whose StandardScaler usage we audit for train-only fitting
# why: a scaler fit on the full dataset (train+test) leaks test distribution into training
# how: D4 check walks each script's AST and flags any StandardScaler() not inside a
#      fold loop or fitted on a clearly train-scoped array
SCALER_AUDIT_FILES = [
    NOTEBOOK_DIR / "03_train_models.py",
    NOTEBOOK_DIR / "04_calibration.py",
    NOTEBOOK_DIR / "06_tuning_optuna.py",
]

# what: columns we refuse to use for modeling because they leak the future
# why: each one peeks at information that would not be available at trade time
# how: dropped before any model is fit; tested below (test F3)
FORBIDDEN_LEAKY_COLS = {
    # Causality / future-looking (original v5 set)
    "kyle_lambda_market_static",
    "wallet_funded_by_cex",
    "n_tokentx",
    "wallet_prior_win_rate",
    # P0-11 — direction-encoding pair: jointly determine bet_correct via XOR formula whose
    # mapping flips across market resolution types (catastrophic test-set inversion on
    # single-resolution cohorts; the exam test set is all-NO ceasefires). See foundation.md.
    "side_buy",
    "outcome_yes",
    # P0-12 — direction-encoding aggregates with 30-pt train/test cohort shift
    "market_buy_share_running",
    "taker_directional_purity_in_market",
    "taker_position_size_before_trade",
    "consensus_strength",
    "contrarian_score",
    "contrarian_strength",
    "is_long_shot_buy",
    "taker_yes_share_global",
    "yes_buy_pressure_5min",
    "yes_volume_share_recent_1h",
    "yes_volume_share_recent_5min",
    "order_flow_imbalance_1h",
    "order_flow_imbalance_24h",
    "order_flow_imbalance_5min",
    "signed_oi_autocorr_1h",
    "token_side_skew_5min",
    # P0-8 — absolute-scale features that leak market identity
    "log_time_to_deadline_hours",
    "is_within_5min_of_deadline",
    "is_within_1h_of_deadline",
    "is_within_24h_of_deadline",
    "market_price_vol_last_5min",
    "market_price_vol_last_24h",
    # P0-8 — absolute price levels (cost-derived; leak the market's price benchmark)
    "recent_price_high_1h",
    "recent_price_low_1h",
    "recent_price_range_1h",
    "recent_price_mean_5min",
    "recent_price_mean_1h",
    "recent_price_mean_24h",
    "pre_trade_price_change_5min",
    "pre_trade_price_change_1h",
    "pre_trade_price_change_24h",
    "distance_from_boundary",
    "log_payoff_if_correct",
    "risk_reward_ratio_pre",
    # Market price benchmark — explicitly excluded from features so p_hat stays
    # independent of market belief; retained in the parquet for backtest cost math.
    "pre_trade_price",
}

# what: row counts we expect from the team's 2026-04-29 release
# why: a short-circuit check that we are reading the right file
EXPECTED_TRAIN_ROWS = 1_114_003
EXPECTED_TEST_ROWS = 257_177
EXPECTED_TOTAL_ROWS = EXPECTED_TRAIN_ROWS + EXPECTED_TEST_ROWS

# what: backtest-only context omitted from the modeling parquet
# why: realistic backtests need true trade USD and corrected YES-normalized price,
#      but these raw fields are not modeling features and must stay out of X
BACKTEST_CONTEXT = DATA_DIR / "backtest_context.parquet"
BACKTEST_CONTEXT_REQUIRED_COLS = {
    "split", "row_in_split", "market_id", "timestamp", "usd_amount",
    "price", "token_amount", "pre_yes_price_corrected",
    "taker", "taker_direction", "nonusdc_side",
}

# what: timestamps of the two real-world events that bracket the test cohort
# why: any trade timestamped after these is leakage (post-event)
STRIKE_EVENT_UTC = pd.Timestamp("2026-02-28T06:35:00", tz="UTC").timestamp()
CEASEFIRE_EVENT_UTC = pd.Timestamp("2026-04-07T23:59:59", tz="UTC").timestamp()


### Script 01 — Load, Split, And Basic Checks

Reads the consolidated parquet, splits rows by cohort, and checks row counts, class balance, event timing, forbidden columns, pre-trade prices, and backtest alignment.


In [10]:
def load_consolidated() -> pd.DataFrame:
    """Load the single source-of-truth parquet."""
    # what: locate the data file
    src = DATA_DIR / "consolidated_modeling_data.parquet"
    if not src.exists():
        raise SystemExit(
            f"Dataset not found at {src}\n"
            "See submission/data/README.md for the download instructions."
        )
    # how: pandas reads the parquet directly into a DataFrame
    df = pd.read_parquet(src)
    print(f"  loaded {len(df):,} rows × {len(df.columns)} cols from {src.name}")
    return df


def split_train_test(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Use the `split` column to separate train and test (cohort-disjoint by market)."""
    # what: the `split` column was assigned upstream by market cohort, not by random shuffle
    # why: random shuffle would leak info from the same market across train and test
    # how: simple boolean masking
    train = df[df["split"] == "train"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)
    print(f"  train: {len(train):,}  test: {len(test):,}")
    return train, test


def check_row_counts(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """Test S1 — row counts match the released contract."""
    # what: hard contract from the release manifest
    # why: catches accidental file swap or partial download
    ok = (len(train) == EXPECTED_TRAIN_ROWS) and (len(test) == EXPECTED_TEST_ROWS)
    return {"name": "S1_row_counts", "pass": ok,
            "train_rows": len(train), "test_rows": len(test),
            "expected_train": EXPECTED_TRAIN_ROWS, "expected_test": EXPECTED_TEST_ROWS}


def check_class_balance(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """Test S2 — target is roughly balanced (no severe imbalance to handle)."""
    # what: report positive-class rate; the dataset is ~50/50 by construction
    # why: imbalance would force SMOTE/ADASYN; balanced data lets us focus on signal
    train_pos = float(train[TARGET].mean())
    test_pos = float(test[TARGET].mean())
    return {"name": "S2_class_balance", "pass": abs(train_pos - 0.5) < 0.05,
            "train_pos_rate": train_pos, "test_pos_rate": test_pos}


def check_no_post_event_leakage(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """Test C1 — no trades timestamped after the cohort's resolving event."""
    # what: train cohort ends at the strike event; test cohort ends at the ceasefire
    # why: a single post-event trade would let the model peek at the resolved outcome
    train_max = float(train["timestamp"].max())
    test_max = float(test["timestamp"].max())
    train_ok = train_max < STRIKE_EVENT_UTC
    test_ok = test_max < CEASEFIRE_EVENT_UTC
    return {"name": "C1_no_post_event_trades", "pass": train_ok and test_ok,
            "train_max_iso": str(pd.to_datetime(train_max, unit="s", utc=True)),
            "test_max_iso": str(pd.to_datetime(test_max, unit="s", utc=True))}


def check_no_forbidden_columns(df: pd.DataFrame) -> dict:
    """Test F3 — forbidden columns are filtered out of the feature list, even if present in the data."""
    # what: the consolidated parquet keeps these columns for traceback completeness, but we never feed them to a model
    # how: get_feature_cols() below excludes them; this check just confirms the exclusion happened
    # why: documenting the policy in the leakage report makes the safety-by-construction visible
    present_in_data = sorted(set(df.columns) & FORBIDDEN_LEAKY_COLS)
    feature_cols_after_exclusion = set(get_feature_cols(df))
    leak_into_features = sorted(feature_cols_after_exclusion & FORBIDDEN_LEAKY_COLS)
    return {"name": "F3_forbidden_cols_excluded_from_features",
            "pass": len(leak_into_features) == 0,
            "present_in_data_but_excluded": present_in_data,
            "would_leak_into_features": leak_into_features,
            "forbidden_set": sorted(FORBIDDEN_LEAKY_COLS)}


def check_pre_trade_price(test: pd.DataFrame) -> dict:
    """Test D1 — pre_trade_price is the previous trade's per-token price (not the current one)."""
    # what: spot-check the upstream feature pre_trade_price by re-deriving it ourselves
    # how: group by market_id, sort by timestamp, take price.shift(1); the first trade in each market gets 0.5
    # why: a one-bar shift error here would give the model the trade's own price as a "feature"
    if "pre_trade_price" not in test.columns or "price" not in test.columns:
        return {"name": "D1_pre_trade_price", "pass": True, "skipped": "raw price column not in dataset"}
    cols = ["market_id", "timestamp", "price", "pre_trade_price"]
    df = test[cols].copy()
    df["market_id"] = df["market_id"].astype(str)
    df = df.sort_values(["market_id", "timestamp"]).reset_index(drop=True)
    expected = df.groupby("market_id")["price"].shift(1).fillna(0.5).values
    actual = df["pre_trade_price"].values
    match_rate = float(np.mean(np.abs(actual - expected) < 1e-6))
    return {"name": "D1_pre_trade_price", "pass": match_rate >= 0.995,
            "match_rate": match_rate}


def check_backtest_context(df: pd.DataFrame) -> dict:
    """Test B1 — backtest context exists and aligns row-for-row with the modeling data."""
    # what: the modeling parquet intentionally omits raw trading fields; this sidecar supplies them
    # why: silent fallbacks in the backtest would make ROI and liquidity assumptions non-reproducible
    if not BACKTEST_CONTEXT.exists():
        return {"name": "B1_backtest_context", "pass": False,
                "reason": f"missing {BACKTEST_CONTEXT.name}"}
    ctx = pd.read_parquet(BACKTEST_CONTEXT)
    missing = sorted(BACKTEST_CONTEXT_REQUIRED_COLS - set(ctx.columns))
    if missing:
        return {"name": "B1_backtest_context", "pass": False,
                "reason": "missing required columns", "missing_columns": missing}
    if len(ctx) != len(df):
        return {"name": "B1_backtest_context", "pass": False,
                "reason": "row count mismatch", "context_rows": len(ctx),
                "model_rows": len(df)}

    split_checks = {}
    for split in ("train", "test"):
        model_sub = df[df["split"] == split][["market_id", "timestamp"]].reset_index(drop=True)
        ctx_sub = ctx[ctx["split"] == split].reset_index(drop=True)
        expected_row = np.arange(len(model_sub))
        row_ok = np.array_equal(ctx_sub["row_in_split"].to_numpy(), expected_row)
        key_ok = (
            model_sub["market_id"].astype(str).to_numpy() == ctx_sub["market_id"].astype(str).to_numpy()
        ).all() and (
            model_sub["timestamp"].to_numpy() == ctx_sub["timestamp"].to_numpy()
        ).all()
        price_ok = ctx_sub["pre_yes_price_corrected"].between(0, 1).all()
        usd_ok = np.isfinite(ctx_sub["usd_amount"]).all() and (ctx_sub["usd_amount"] >= 0).all()
        split_checks[split] = {
            "rows": int(len(ctx_sub)),
            "row_in_split_ok": bool(row_ok),
            "market_timestamp_alignment_ok": bool(key_ok),
            "pre_yes_price_in_0_1": bool(price_ok),
            "usd_amount_finite_nonnegative": bool(usd_ok),
        }
    ok = all(all(v for k, v in c.items() if k != "rows") for c in split_checks.values())
    return {"name": "B1_backtest_context", "pass": bool(ok),
            "context_file": BACKTEST_CONTEXT.name, "checks": split_checks}


### Script 01 — Scaler Leakage Audit

AST-based audit that flags unsafe `StandardScaler` fitting patterns in companion script files when they exist.


In [11]:
def _enclosing_func(tree: ast.AST, target: ast.AST) -> ast.FunctionDef | None:
    """Return the FunctionDef node that directly contains `target`, or None."""
    # what: walk the AST to find which function owns the target node
    # why: we need the function scope to check for fold loops and train-subset fits
    # how: iterate all FunctionDef nodes, inner-walk each; first match wins
    candidate: ast.FunctionDef | None = None
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            for inner in ast.walk(node):
                if inner is target:
                    candidate = node
                    break
            if candidate is not None:
                break
    return candidate


def _has_fold_loop(func: ast.FunctionDef) -> bool:
    """True if `func` body contains a recognisable cross-validation split call.

    Recognised patterns:
      - `<x>.split(...)` where x is any CV splitter (KFold, GroupKFold, etc.)
      - any `.split(...)` attribute call — conservative but sufficient
    """
    # what: look for .split(...) calls anywhere in the function body
    # why: a CV splitter's .split() is the canonical fold-loop marker
    # how: walk all Call nodes; check for an Attribute named "split"
    for node in ast.walk(func):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
            if node.func.attr == "split":
                return True
    return False


def _fits_only_train_subset(func: ast.FunctionDef) -> bool:
    """True if every fit/fit_transform call in `func` targets a train-scoped array.

    Recognises:
      - X.iloc[tr_idx] or X[tr_idx]  →  ast.Subscript
      - X_train, X_tr                →  ast.Name ending in _train/_tr
    Returns True if at least one qualifying pattern is found, or if no
    fit/fit_transform calls exist at all (scaler instantiated but unused — not a risk).
    """
    # what: check that fit_transform is called on a train-restricted subset
    # why: fit_transform on unsliced X uses test rows and leaks the test distribution
    # how: inspect the first argument of each fit/fit_transform call
    seen_any = False
    for node in ast.walk(func):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute):
            if node.func.attr in ("fit_transform", "fit"):
                if not node.args:
                    continue
                seen_any = True
                arg0 = node.args[0]
                # X.iloc[...] or X[...]
                if isinstance(arg0, ast.Subscript):
                    return True
                # X.iloc[...].something (Attribute of a Subscript)
                if isinstance(arg0, ast.Attribute) and isinstance(
                    arg0.value, ast.Subscript
                ):
                    return True
                # X_train, X_tr, etc.
                if isinstance(arg0, ast.Name) and (
                    arg0.id.endswith("_train") or arg0.id.endswith("_tr")
                ):
                    return True
    # no fit calls at all — scaler present but not fitted here; not a leak risk
    return not seen_any


def check_scaler_refit_per_fold() -> dict:
    """Test D4 — every StandardScaler() is inside a CV fold loop or fits only a train subset.

    what: AST-walk each modelling script and flag any StandardScaler() instantiation
          that is neither inside a function with a .split() call nor fitted on a
          clearly train-scoped array (subscripted, iloc'd, or a Name ending _train/_tr).
    why:  a scaler fit on the full dataset before the CV split leaks test-set
          distribution statistics into the training signal.
    how:  parse each file with ast.parse, find all StandardScaler() Call nodes,
          look up their enclosing function, then apply _has_fold_loop and
          _fits_only_train_subset heuristics.
    """
    findings = []
    suspicious = []

    for path in SCALER_AUDIT_FILES:
        if not path.exists():
            findings.append({"file": path.name, "reason": "file_not_found", "ok": True})
            continue
        try:
            tree = ast.parse(path.read_text())
        except SyntaxError as e:
            suspicious.append({"file": path.name, "reason": f"SyntaxError: {e}"})
            continue

        for node in ast.walk(tree):
            if (
                isinstance(node, ast.Call)
                and isinstance(node.func, ast.Name)
                and node.func.id == "StandardScaler"
            ):
                line_no = getattr(node, "lineno", -1)
                func = _enclosing_func(tree, node)
                func_name = func.name if func else "<module>"
                ok_fold = _has_fold_loop(func) if func else False
                ok_subset = _fits_only_train_subset(func) if func else False
                ok = ok_fold or ok_subset
                findings.append(
                    {
                        "file": path.name,
                        "line": line_no,
                        "function": func_name,
                        "has_fold_loop": ok_fold,
                        "fits_train_subset": ok_subset,
                        "ok": ok,
                    }
                )
                if not ok:
                    suspicious.append(
                        {
                            "file": path.name,
                            "line": line_no,
                            "function": func_name,
                            "reason": (
                                "StandardScaler outside CV loop AND "
                                "fit_transform not on a train-subset"
                            ),
                        }
                    )

    return {
        "name": "D4_scaler_refit_per_fold",
        "pass": len(suspicious) == 0,
        "findings": findings,
        "suspicious": suspicious,
    }


### Script 01 — Wallet Bisection Check

Checks that wallet-level features are computed without crossing the train/test boundary.


In [12]:
def check_wallet_bisection(test: pd.DataFrame) -> dict:
    """Test W1 — wallet features re-derived from raw enrichment match the parquet values.

    what: for a 500-row sample of the test set, re-compute three wallet features
          (wallet_polygon_age_at_t_days, wallet_n_inbound_at_t, wallet_n_cex_deposits_at_t)
          from wallet_enrichment.parquet using only events with timestamp < t, then
          compare against the values stored in the modelling parquet.
    why:  detects any off-by-one shift or future-looking join in the wallet feature
          engineering step; a mismatch here would mean the wallet features leak
          post-trade data.
    how:  read enrichment with fetch_status=='ok', build wallet→info lookup dict,
          sample 500 rows with seed=42, recompute using np.searchsorted (bisect-left),
          require >=99% match rate to pass.
    """
    wallet_enrich_path = DATA_DIR / "wallet_enrichment.parquet"

    # what: graceful skip when the file is absent from submission/data/
    # why: the upstream dev already verified this on the full dataset;
    #      in the submission boundary we document as skipped rather than failing
    if not wallet_enrich_path.exists():
        return {
            "name": "W1_wallet_bisection",
            "pass": True,
            "skipped": "wallet_enrichment.parquet not in submission/data/",
        }

    required_cols = [
        "wallet_polygon_age_at_t_days",
        "wallet_n_inbound_at_t",
        "wallet_n_cex_deposits_at_t",
    ]
    missing_cols = [c for c in required_cols if c not in test.columns]
    if missing_cols:
        return {
            "name": "W1_wallet_bisection",
            "pass": True,
            "skipped": f"wallet feature columns not in test set: {missing_cols}",
        }

    if "taker" not in test.columns:
        return {
            "name": "W1_wallet_bisection",
            "pass": True,
            "skipped": "taker column not in test set",
        }

    # what: load enrichment data; restrict to successfully fetched wallets
    enrich = pd.read_parquet(wallet_enrich_path)
    enrich = enrich[enrich["fetch_status"] == "ok"]

    # what: build a per-wallet lookup dict with sorted timestamp arrays
    # how: inbound_ts and cex_deposit_ts are list-of-int columns in the enrichment parquet
    idx: dict[str, dict] = {}
    for _, r in enrich.iterrows():
        idx[str(r["wallet"]).lower()] = {
            "polygon_first_tx_ts": (
                int(r["polygon_first_tx_ts"])
                if not pd.isna(r["polygon_first_tx_ts"])
                else None
            ),
            "inbound_ts": (
                np.asarray(r["inbound_ts"], dtype=np.int64)
                if len(r["inbound_ts"])
                else np.array([], dtype=np.int64)
            ),
            "cex_deposit_ts": (
                np.asarray(r["cex_deposit_ts"], dtype=np.int64)
                if len(r["cex_deposit_ts"])
                else np.array([], dtype=np.int64)
            ),
        }

    # what: sample 500 test rows uniformly with a fixed seed for reproducibility
    rng = np.random.RandomState(42)
    n_sample = min(500, len(test))
    sample_idx = rng.choice(len(test), size=n_sample, replace=False)

    n_checked = 0
    n_match = 0
    mismatches: list[dict] = []

    for i in sample_idx:
        wallet = str(test.iloc[i]["taker"]).lower()
        info = idx.get(wallet)
        if info is None:
            continue
        n_checked += 1
        t = int(test.iloc[i]["timestamp"])

        # what: re-derive each feature using only events strictly before timestamp t
        # how: np.searchsorted with side='left' gives count of events < t
        if info["polygon_first_tx_ts"] is None:
            exp_age = float("nan")
        else:
            exp_age = max(0, t - info["polygon_first_tx_ts"]) / 86400.0

        exp_inbound = (
            int(np.searchsorted(info["inbound_ts"], t, side="left"))
            if len(info["inbound_ts"])
            else 0
        )
        exp_cex = (
            int(np.searchsorted(info["cex_deposit_ts"], t, side="left"))
            if len(info["cex_deposit_ts"])
            else 0
        )

        act_age = test.iloc[i]["wallet_polygon_age_at_t_days"]
        act_inbound = test.iloc[i]["wallet_n_inbound_at_t"]
        act_cex = test.iloc[i]["wallet_n_cex_deposits_at_t"]

        ok_age = (np.isnan(exp_age) and pd.isna(act_age)) or (
            not np.isnan(exp_age)
            and not pd.isna(act_age)
            and abs(float(act_age) - exp_age) < 1e-3
        )
        ok_inbound = float(act_inbound) == float(exp_inbound)
        ok_cex = float(act_cex) == float(exp_cex)

        if ok_age and ok_inbound and ok_cex:
            n_match += 1
        elif len(mismatches) < 10:
            mismatches.append(
                {
                    "row": int(i),
                    "wallet": wallet[:10] + "...",
                    "ts": t,
                    "age_exp": float(exp_age) if not np.isnan(exp_age) else None,
                    "age_act": float(act_age) if not pd.isna(act_age) else None,
                    "inbound_exp": exp_inbound,
                    "inbound_act": float(act_inbound),
                    "cex_exp": exp_cex,
                    "cex_act": float(act_cex),
                }
            )

    match_rate = n_match / n_checked if n_checked > 0 else 0.0
    passed = n_checked == 0 or match_rate >= 0.99

    return {
        "name": "W1_wallet_bisection",
        "pass": bool(passed),
        "n_sampled": int(n_sample),
        "n_checked": n_checked,
        "n_match": n_match,
        "match_rate": float(match_rate),
        "mismatches_sample": mismatches,
    }


### Script 01 — Feature Export And Stage Entrypoint

Builds the canonical feature list, runs every check, writes reports, and hard-stops if a check fails.


In [13]:
def get_feature_cols(df: pd.DataFrame) -> list[str]:
    """Return the list of columns that are actually features (not meta, not target, not forbidden)."""
    # what: filter the column list down to modeling features
    # why: this is the canonical feature list every downstream script should use
    excluded = META_COLS | {TARGET} | FORBIDDEN_LEAKY_COLS
    return sorted([c for c in df.columns if c not in excluded])


def main_01() -> int:
    # what: header so terminal output is easy to scan
    print("=" * 60)
    print("Stage 1 — Load data, run leakage checks, save feature list")
    print("=" * 60)

    # what: ensure the output folder exists for our reports
    out_dir = OUTPUTS_DIR / "data"
    out_dir.mkdir(parents=True, exist_ok=True)

    # what: load + split
    df = load_consolidated()
    train, test = split_train_test(df)

    # what: run all leakage / sanity checks and collect results in one report
    # why: a single JSON file is easier for the teacher to inspect than terminal scrollback
    checks = [
        check_row_counts(train, test),
        check_class_balance(train, test),
        check_no_post_event_leakage(train, test),
        check_no_forbidden_columns(df),
        check_pre_trade_price(test),
        check_scaler_refit_per_fold(),
        check_wallet_bisection(test),
        check_backtest_context(df),
    ]
    n_pass = sum(1 for c in checks if c["pass"])
    for c in checks:
        flag = "PASS" if c["pass"] else "FAIL"
        print(f"  [{flag}] {c['name']}")
    print(f"  -> {n_pass}/{len(checks)} checks passed")

    # what: persist the leakage report next to the modeling outputs
    leak_path = out_dir / "leakage_report.json"
    leak_path.write_text(json.dumps({"checks": checks, "n_pass": n_pass}, indent=2))
    print(f"  saved leakage report -> {leak_path.relative_to(OUTPUTS_DIR.parent)}")

    # what: write the final feature list used by every downstream script
    feature_cols = get_feature_cols(df)
    fc_path = out_dir / "feature_cols.json"
    fc_path.write_text(json.dumps(feature_cols, indent=2))
    print(f"  saved {len(feature_cols)} feature names -> {fc_path.relative_to(OUTPUTS_DIR.parent)}")

    # what: hard-stop the pipeline if any leakage check failed
    # why: silently continuing would invalidate every model trained downstream
    if n_pass != len(checks):
        print("\nLeakage check failed. Refusing to continue.")
        return 1
    print("\nStage 1 complete. Proceed to 02_features.py.")
    return 0


### Script 01 — Run

Inputs: `data/consolidated_modeling_data.parquet` and `data/backtest_context.parquet`. Outputs: feature list and leakage report under `outputs/data/`.


In [14]:
np.random.seed(RANDOM_SEED)
main_01()


Stage 1 — Load data, run leakage checks, save feature list
  loaded 1,371,180 rows × 87 cols from consolidated_modeling_data.parquet
  train: 1,114,003  test: 257,177
  [PASS] S1_row_counts
  [PASS] S2_class_balance
  [PASS] C1_no_post_event_trades
  [PASS] F3_forbidden_cols_excluded_from_features
  [PASS] D1_pre_trade_price
  [PASS] D4_scaler_refit_per_fold
  [PASS] W1_wallet_bisection
  [PASS] B1_backtest_context
  -> 8/8 checks passed
  saved leakage report -> outputs/data/leakage_report.json
  saved 43 feature names -> outputs/data/feature_cols.json

Stage 1 complete. Proceed to 02_features.py.


0

## Script 02 — Features, Scaler, Isolation Forest

Documents feature taxonomy, fits a train-only `StandardScaler`, and writes Isolation Forest anomaly scores. Runtime is about 5 minutes.


### Script 02 — Imports And Constants

Loads the packages and constants needed for feature taxonomy, scaling, and anomaly scoring.


In [15]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# what: pull shared paths and the project random seed
TARGET = "bet_correct"
META_COLS = ["split", "market_id", "ts_dt", "timestamp"]


### Script 02 — Load Data And Build Taxonomy

Reads the feature list from Script 01 and groups feature names into report-friendly categories.


In [16]:
def load_data() -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Load the parquet, split train/test, read the feature list saved by 01_data_prep."""
    # what: read everything 01_data_prep produced + the dataset
    df = pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet")
    feature_cols = json.loads((OUTPUTS_DIR / "data" / "feature_cols.json").read_text())
    train = df[df["split"] == "train"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)
    print(f"  loaded train={len(train):,} test={len(test):,} features={len(feature_cols)}")
    return train, test, feature_cols


def build_taxonomy(feature_cols: list[str]) -> dict:
    """Group features by source so the report's methodology section can describe them clearly."""
    # what: bucket each feature name into a thematic group for the methodology table
    # how: simple prefix / substring rules; the upstream naming convention makes this safe
    # why: lets the report cite "X trade-microstructure features, Y wallet features, ..."
    taxonomy: dict[str, list[str]] = {
        "wallet": [], "market": [], "trade": [], "microstructure": [], "history": [], "other": [],
    }
    for c in feature_cols:
        if c.startswith("wallet_") or c == "days_from_first_usdc_to_t":
            taxonomy["wallet"].append(c)
        elif c.startswith("market_") or "deadline" in c or "resolution" in c:
            taxonomy["market"].append(c)
        elif "kyle" in c or "spread" in c or "depth" in c or "imbalance" in c or "lambda" in c:
            taxonomy["microstructure"].append(c)
        elif "taker_" in c or "wallet_prior" in c or "history" in c:
            taxonomy["history"].append(c)
        elif c in {"price", "pre_trade_price", "pre_yes_price_corrected", "token_amount",
                   "usd_amount", "side_buy", "outcome_yes", "nonusdc_side"}:
            taxonomy["trade"].append(c)
        else:
            taxonomy["other"].append(c)
    counts = {k: len(v) for k, v in taxonomy.items()}
    print("  feature taxonomy:", counts)
    return {"counts": counts, "groups": taxonomy}


### Script 02 — Scaler And Anomaly Score

Fits the scaler on train rows only, then trains/scales Isolation Forest scores for train and test rows.


In [17]:
def fit_scaler(train: pd.DataFrame, feature_cols: list[str]) -> StandardScaler:
    """Fit StandardScaler on the train split only (no test leakage)."""
    # what: replace any inf with NaN, then fill NaN with 0, then fit
    # why: linear and MLP models need standardised inputs; trees do not but the scaler is harmless
    # how: fit on train rows ONLY — fitting on test would leak distribution info
    X_train = train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    scaler = StandardScaler().fit(X_train)
    print(f"  scaler fit on {len(X_train):,} rows × {len(feature_cols)} features")
    return scaler


def add_iso_forest_score(train: pd.DataFrame, test: pd.DataFrame,
                          feature_cols: list[str], scaler: StandardScaler) -> pd.DataFrame:
    """Train an Isolation Forest on train, score every row (train + test).

    Curriculum link: outlier detection (lecture 7). Hypothesis: unusual trades sit
    further from the joint feature distribution than retail trades, so a higher
    anomaly score should correlate with bet_correct.
    """
    # what: prepare the matrices in the same way as the supervised models
    X_train = scaler.transform(
        train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values)
    X_test = scaler.transform(
        test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values)

    # what: 200 trees, sub-sampled to 50k for speed; default contamination
    # how: IsoForest splits points randomly until isolated; faster isolation = more anomalous
    # why: pure-unsupervised signal, decoupled from the supervised target
    print("  fitting Isolation Forest (200 trees, sample 50k) ...")
    iso = IsolationForest(n_estimators=200, max_samples=min(50_000, len(X_train)),
                          contamination="auto", random_state=RANDOM_SEED, n_jobs=-1)
    iso.fit(X_train)

    # what: score_samples returns higher = more normal; we negate so higher = more anomalous
    # why: easier to interpret as "anomaly score" for the report
    train_score = -iso.score_samples(X_train)
    test_score = -iso.score_samples(X_test)
    s_min, s_max = train_score.min(), train_score.max()
    train_scaled = (train_score - s_min) / (s_max - s_min + 1e-9)
    test_scaled = np.clip((test_score - s_min) / (s_max - s_min + 1e-9), 0, 1)

    # what: report the correlation between anomaly score and target as a quick sanity check
    train_corr = float(np.corrcoef(train_scaled, train[TARGET])[0, 1])
    test_corr = float(np.corrcoef(test_scaled, test[TARGET])[0, 1])
    print(f"  anomaly-target correlation: train {train_corr:+.4f}  test {test_corr:+.4f}")

    # what: stack (split, market_id, timestamp, anomaly_score) for downstream join
    rows = []
    for split, sub, scores in [("train", train, train_scaled), ("test", test, test_scaled)]:
        rows.append(pd.DataFrame({
            "split": split,
            "market_id": sub["market_id"].astype(str).values,
            "timestamp": sub["timestamp"].values,
            "anomaly_score": scores,
        }))
    return pd.concat(rows, ignore_index=True)


### Script 02 — Stage Entrypoint

Writes `feature_taxonomy.json`, `scaler.joblib`, and `iso_forest_scores.parquet`.


In [18]:
def main_02() -> int:
    print("=" * 60)
    print("Stage 2 — Feature taxonomy, scaler, anomaly score")
    print("=" * 60)
    out_dir = OUTPUTS_DIR / "data"
    out_dir.mkdir(parents=True, exist_ok=True)

    # what: load + taxonomy
    train, test, feature_cols = load_data()
    taxonomy = build_taxonomy(feature_cols)
    (out_dir / "feature_taxonomy.json").write_text(json.dumps(taxonomy, indent=2))

    # what: fit + persist scaler
    scaler = fit_scaler(train, feature_cols)
    joblib.dump(scaler, out_dir / "scaler.joblib")

    # what: build anomaly score
    iso_scores = add_iso_forest_score(train, test, feature_cols, scaler)
    iso_scores.to_parquet(out_dir / "iso_forest_scores.parquet", index=False)

    print(f"\nStage 2 complete. {len(feature_cols)} features ready for modeling.")
    return 0


### Script 02 — Run

Inputs: Script 01 feature list and consolidated data. Outputs: taxonomy, scaler, and anomaly-score parquet under `outputs/data/`.


In [19]:
np.random.seed(RANDOM_SEED)
main_02()


Stage 2 — Feature taxonomy, scaler, anomaly score
  loaded train=1,114,003 test=257,177 features=43
  feature taxonomy: {'wallet': 11, 'market': 1, 'trade': 0, 'microstructure': 0, 'history': 10, 'other': 21}
  scaler fit on 1,114,003 rows × 43 features
  fitting Isolation Forest (200 trees, sample 50k) ...
  anomaly-target correlation: train +0.0065  test -0.0057

Stage 2 complete. 43 features ready for modeling.


0

## Script 03 — Train And Compare Models

Trains the supervised model lineup with grouped cross-validation, saves OOF/test predictions, and writes comparison/complexity tables. Runtime is about 60 minutes.


### Script 03 — Imports And Shared Metrics

Defines scoring helpers and loads the target/feature matrices for modeling.


In [20]:
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import (HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# what: shared paths/seeds
TARGET = "bet_correct"


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------

def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    """ECE = weighted mean | empirical_pos_rate - mean_predicted_prob | per bin."""
    # what: bucket predictions into n_bins, compare per-bucket avg-prob to per-bucket actual rate
    # why: AUC ignores calibration; ECE captures whether 0.8 really means "80% chance"
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins[1:-1], right=False)
    ece = 0.0
    for b in range(n_bins):
        mask = idx == b
        if not mask.any():
            continue
        ece += mask.mean() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)


def metric_block(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    """Compute the standard scoring block for one prediction array."""
    return {
        "auc": float(roc_auc_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "ece": expected_calibration_error(y_true, y_prob),
        "n": int(len(y_true)),
        "pos_rate": float(y_true.mean()),
    }


def load_xy() -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.DataFrame, pd.Series, list[str]]:
    """Load data and the canonical feature list saved by 01_data_prep."""
    # what: read the consolidated parquet + the feature list saved by 01_data_prep
    df = pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet")
    feature_cols = json.loads((OUTPUTS_DIR / "data" / "feature_cols.json").read_text())

    # what: replace inf with nan, then nan with 0 — same convention used end-to-end
    # why: tree models tolerate nan but linear/MLP need finite floats
    train = df[df["split"] == "train"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)
    X_train = train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_train = train[TARGET].astype(int)
    y_test = test[TARGET].astype(int)
    g_train = train["market_id"]
    print(f"  train={len(X_train):,}  test={len(X_test):,}  features={len(feature_cols)}")
    return X_train, y_train, g_train, X_test, y_test, feature_cols, test


### Script 03 — Cross-Validation And Evaluation

Builds OOF predictions, fits final train models, saves per-model artifacts, and records headline metrics.


In [21]:
def cv_oof(make_estimator, X: pd.DataFrame, y: pd.Series, groups: pd.Series, scale: bool) -> tuple[np.ndarray, list[float]]:
    """Run 5-fold GroupKFold CV, return out-of-fold predictions + per-fold AUCs."""
    # what: predefined splitter; group=market_id keeps each market in one fold only
    # why: random split would let the same market appear in both train and val of one fold => leakage
    gkf = GroupKFold(n_splits=N_FOLDS)
    oof = np.zeros(len(y), dtype=float)
    fold_aucs: list[float] = []
    for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
        # what: refit scaler PER FOLD on training rows only (no test leakage)
        if scale:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X.iloc[tr_idx])
            X_va = scaler.transform(X.iloc[va_idx])
        else:
            X_tr = X.iloc[tr_idx].values
            X_va = X.iloc[va_idx].values
        clf = make_estimator()
        clf.fit(X_tr, y.iloc[tr_idx])
        # what: get probabilities; if model has only decision_function, min-max it within fold
        if hasattr(clf, "predict_proba"):
            preds = clf.predict_proba(X_va)[:, 1]
        else:
            raw = clf.decision_function(X_va)
            preds = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
        oof[va_idx] = preds
        fold_aucs.append(float(roc_auc_score(y.iloc[va_idx], preds)))
        print(f"    fold {fold_idx}/{N_FOLDS}: val AUC={fold_aucs[-1]:.4f}")
    return oof, fold_aucs


def fit_final(make_estimator, X_train: pd.DataFrame, y_train: pd.Series,
              X_test: pd.DataFrame, scale: bool):
    """Fit one final model on the FULL train set, return raw test predictions + the fitted clf."""
    # what: fit one final model on all training rows for the headline test scoring
    if scale:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
    else:
        scaler, X_tr, X_te = None, X_train.values, X_test.values
    clf = make_estimator()
    clf.fit(X_tr, y_train)
    if hasattr(clf, "predict_proba"):
        raw = clf.predict_proba(X_te)[:, 1]
    else:
        rawd = clf.decision_function(X_te)
        raw = (rawd - rawd.min()) / (rawd.max() - rawd.min() + 1e-9)
    return clf, scaler, raw


def evaluate_model(name: str, make_estimator, X_train, y_train, g_train, X_test, y_test,
                    scale: bool) -> dict:
    """End-to-end: CV + final fit + save preds + return summary."""
    print(f"\n[{name}]")
    out_dir = OUTPUTS_DIR / "models" / name
    out_dir.mkdir(parents=True, exist_ok=True)

    # what: 5-fold OOF for downstream calibration
    oof, fold_aucs = cv_oof(make_estimator, X_train, y_train, g_train, scale=scale)
    cv_oof_auc = float(roc_auc_score(y_train, oof))
    print(f"  OOF AUC = {cv_oof_auc:.4f}  (folds {[f'{a:.3f}' for a in fold_aucs]})")
    np.save(out_dir / "preds_oof.npy", oof.astype("float32"))

    # what: full-train fit + test scoring (raw probabilities only; calibration in 04_)
    clf, _, raw = fit_final(make_estimator, X_train, y_train, X_test, scale=scale)
    np.savez_compressed(out_dir / "preds_test.npz", raw=raw.astype("float32"))
    test_metrics = metric_block(y_test.values, raw)

    # what: assemble + persist summary
    summary = {
        "model": name,
        "cv_oof_auc": cv_oof_auc,
        "cv_fold_aucs": fold_aucs,
        "cv_fold_mean": float(np.mean(fold_aucs)),
        "cv_fold_std": float(np.std(fold_aucs)),
        "test_raw": test_metrics,
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "n_features": int(X_train.shape[1]),
    }
    (out_dir / "metrics.json").write_text(json.dumps(summary, indent=2))
    print(f"  test AUC (raw) = {test_metrics['auc']:.4f}  Brier = {test_metrics['brier']:.4f}")
    return summary


### Script 03 — PCA And Complexity Benchmark

Chooses the PCA component count and measures fit time, prediction latency, and model-size proxies.


In [22]:
def pca_elbow_k(X_train: pd.DataFrame, max_components: int = 50) -> int:
    """Pick K via the geometric-elbow method on the cumulative variance curve."""
    # what: standardise, fit PCA with k_max components, find the elbow
    # why: principled K choice (no magic 0.95 threshold) — perpendicular distance from chord
    scaler = StandardScaler()
    X = scaler.fit_transform(X_train)
    k_max = min(max_components, X.shape[1])
    pca = PCA(n_components=k_max, random_state=RANDOM_SEED).fit(X)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    xs = np.arange(1, k_max + 1, dtype=float)
    p1, p2 = np.array([xs[0], cumvar[0]]), np.array([xs[-1], cumvar[-1]])
    line_vec = p2 - p1
    line_len = np.linalg.norm(line_vec)
    points = np.column_stack([xs, cumvar])
    # how: perpendicular distance from each (k, cumvar[k]) to the chord between endpoints
    dists = np.abs(np.cross(line_vec, points - p1)) / line_len
    return int(max(2, xs[int(np.argmax(dists))]))


# ----------------------------------------------------------------------------
# Complexity benchmark (REQUIRED by guidelines: model complexity vs baseline)
# ----------------------------------------------------------------------------

def complexity_proxy(model) -> int:
    """Return a parameter / size proxy for the fitted model."""
    # what: each model family stores complexity differently; report a number that scales sensibly
    # why: methodology section needs a concrete "size" number to compare 200-tree RF vs 64-unit MLP
    if hasattr(model, "coef_"):                      # logistic regression
        return int(model.coef_.size)
    if hasattr(model, "estimators_"):                # bagged trees / forests
        return int(sum(t.tree_.node_count for t in model.estimators_))
    if hasattr(model, "_predictors"):                # HistGBM
        return int(sum(p[0].nodes.size for p in model._predictors))
    if hasattr(model, "tree_"):                      # single decision tree
        return int(model.tree_.node_count)
    if hasattr(model, "coefs_"):                     # MLPClassifier
        return int(sum(c.size for c in model.coefs_) + sum(b.size for b in model.intercepts_))
    if hasattr(model, "booster_"):                   # LightGBM
        return int(model.booster_.num_trees())
    return -1


def benchmark_complexity(make_estimator, X_train, y_train, X_test, scale: bool, repeats: int = 3) -> dict:
    """Measure fit time, predict time per 1k rows, and parameter count."""
    # what: time one fit on the full train set, then time predict_proba over `repeats` runs
    # why: the report's complexity section needs apples-to-apples wall-clock numbers
    if scale:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
    else:
        X_tr, X_te = X_train.values, X_test.values
    t0 = time.time()
    clf = make_estimator()
    clf.fit(X_tr, y_train)
    fit_sec = time.time() - t0
    # how: median of `repeats` predict calls smooths out OS noise
    times = []
    for _ in range(repeats):
        t0 = time.time()
        if hasattr(clf, "predict_proba"):
            clf.predict_proba(X_te)
        else:
            clf.decision_function(X_te)
        times.append((time.time() - t0) * 1000.0 / len(X_te))   # seconds per 1k rows
    return {"fit_sec": float(fit_sec),
            "predict_per_1k_sec": float(np.median(times)),
            "n_params": int(complexity_proxy(clf))}


### Script 03 — Model Factories

Defines the baseline model lineup, including optional LightGBM when the package is installed.


In [23]:
def make_factories(pca_k: int) -> list[tuple]:
    """Return list of (name, factory, scale_required)."""
    # what: every entry is (name, callable that returns a fresh estimator, whether to scale X)
    # why: lambda re-construction guarantees no state leaks across folds
    factories = [
        ("logreg_l2", lambda: LogisticRegression(C=1.0, penalty="l2", class_weight="balanced",
                                                  max_iter=2000, random_state=RANDOM_SEED), True),
        ("logreg_l1", lambda: LogisticRegression(C=0.1, penalty="l1", solver="liblinear",
                                                  class_weight="balanced", max_iter=2000,
                                                  random_state=RANDOM_SEED), True),
        ("decision_tree", lambda: DecisionTreeClassifier(max_depth=8, min_samples_leaf=200,
                                                          class_weight="balanced",
                                                          random_state=RANDOM_SEED), False),
        ("random_forest", lambda: RandomForestClassifier(n_estimators=200, max_depth=10,
                                                          min_samples_leaf=200, n_jobs=N_JOBS,
                                                          class_weight="balanced",
                                                          random_state=RANDOM_SEED), False),
        ("hist_gbm", lambda: HistGradientBoostingClassifier(max_iter=200, max_depth=8,
                                                             learning_rate=0.05,
                                                             class_weight="balanced",
                                                             random_state=RANDOM_SEED), False),
        ("pca_logreg", lambda: Pipeline([
            ("pca", PCA(n_components=pca_k, random_state=RANDOM_SEED)),
            ("lr",  LogisticRegression(C=1.0, penalty="l2", class_weight="balanced",
                                       max_iter=2000, random_state=RANDOM_SEED))]), True),
        ("mlp_sklearn", lambda: MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu",
                                              solver="adam", alpha=1e-4, batch_size=4096,
                                              learning_rate_init=1e-3, max_iter=50,
                                              early_stopping=True, validation_fraction=0.1,
                                              n_iter_no_change=5, random_state=RANDOM_SEED), True),
    ]
    # what: optional LightGBM if installed; common alternative gradient-boosting library
    try:
        import lightgbm as lgb
        factories.append(("lightgbm",
                          lambda: lgb.LGBMClassifier(n_estimators=400, num_leaves=63,
                                                     learning_rate=0.05, min_child_samples=200,
                                                     class_weight="balanced",
                                                     random_state=RANDOM_SEED, verbosity=-1,
                                                     n_jobs=N_JOBS, num_threads=N_JOBS),
                          False))
    except ImportError:
        print("  note: lightgbm not installed; skipping (pip install lightgbm to enable)")
    return factories


### Script 03 — Stage Entrypoint

Runs the model lineup and writes `comparison.csv` plus `complexity.csv`.


In [24]:
def main_03() -> int:
    print("=" * 60)
    print("Stage 3 — Train and compare models")
    print("=" * 60)
    metrics_dir = OUTPUTS_DIR / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    # what: load + size + factories
    X_train, y_train, g_train, X_test, y_test, feature_cols, _test_df = load_xy()
    pca_k = pca_elbow_k(X_train)
    print(f"  PCA elbow K = {pca_k}")
    factories = make_factories(pca_k)

    # what: train every model end-to-end
    summaries = []
    for name, factory, scale in factories:
        try:
            summaries.append(evaluate_model(name, factory, X_train, y_train, g_train,
                                             X_test, y_test, scale=scale))
        except Exception as e:
            print(f"  [{name}] FAILED: {e}")
            summaries.append({"model": name, "error": str(e)})

    # what: comparison table for the report
    rows = [{"model": s["model"],
             "cv_oof_auc": round(s["cv_oof_auc"], 4),
             "test_auc_raw": round(s["test_raw"]["auc"], 4),
             "test_brier_raw": round(s["test_raw"]["brier"], 4),
             "test_ece_raw": round(s["test_raw"]["ece"], 4)}
            for s in summaries if "error" not in s]
    df = pd.DataFrame(rows).sort_values("test_auc_raw", ascending=False)
    df.to_csv(metrics_dir / "comparison.csv", index=False)
    print("\n" + df.to_string(index=False))

    # what: complexity table — fit time + predict latency + parameter count
    print("\nBenchmarking complexity (fit time / predict latency / param count) ...")
    complexity_rows = []
    for name, factory, scale in factories:
        try:
            comp = benchmark_complexity(factory, X_train, y_train, X_test, scale=scale)
            complexity_rows.append({"model": name, **comp})
            print(f"  {name:14s}  fit={comp['fit_sec']:7.2f}s  "
                  f"predict={comp['predict_per_1k_sec']*1000:6.2f}ms/1k  "
                  f"n_params={comp['n_params']:,}")
        except Exception as e:
            print(f"  {name}: skipped ({e})")
    pd.DataFrame(complexity_rows).to_csv(metrics_dir / "complexity.csv", index=False)

    print(f"\nStage 3 complete. Outputs in {metrics_dir.relative_to(OUTPUTS_DIR.parent)}.")
    print("Proceed to 04_calibration.py.")
    return 0


### Script 03 — Run

Inputs: Script 01 feature list and consolidated data. Outputs: model folders under `outputs/models/` plus metrics under `outputs/metrics/`.


In [25]:
np.random.seed(RANDOM_SEED)
main_03()


Stage 3 — Train and compare models
  train=1,114,003  test=257,177  features=43
  PCA elbow K = 16

[logreg_l2]
    fold 1/5: val AUC=0.5809
    fold 2/5: val AUC=0.5237
    fold 3/5: val AUC=0.5472
    fold 4/5: val AUC=0.5967
    fold 5/5: val AUC=0.6085
  OOF AUC = 0.5692  (folds ['0.581', '0.524', '0.547', '0.597', '0.608'])
  test AUC (raw) = 0.5436  Brier = 0.2491

[logreg_l1]
    fold 1/5: val AUC=0.5808
    fold 2/5: val AUC=0.5238
    fold 3/5: val AUC=0.5472
    fold 4/5: val AUC=0.5965
    fold 5/5: val AUC=0.6084
  OOF AUC = 0.5691  (folds ['0.581', '0.524', '0.547', '0.597', '0.608'])
  test AUC (raw) = 0.5436  Brier = 0.2491

[decision_tree]
    fold 1/5: val AUC=0.5943
    fold 2/5: val AUC=0.5396
    fold 3/5: val AUC=0.5888
    fold 4/5: val AUC=0.6619
    fold 5/5: val AUC=0.6550
  OOF AUC = 0.6082  (folds ['0.594', '0.540', '0.589', '0.662', '0.655'])
  test AUC (raw) = 0.5380  Brier = 0.2543

[random_forest]
    fold 1/5: val AUC=0.6263
    fold 2/5: val AUC=0.5454


0

## Script 04 — Calibration, Reliability, CI, Importance

Calibrates model predictions with isotonic regression, generates reliability plots, bootstrap CIs, pairwise tests, permutation importance, SHAP summaries, agreement matrices, and per-market AUC heatmaps. Runtime is about 10 minutes.


### Script 04 — Imports And Constants

Loads plotting/model-evaluation dependencies and sets bootstrap/permutation defaults.


In [26]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

apply_theme()

TARGET = "bet_correct"
N_BOOTSTRAP = 500
PERM_N_REPEATS = 3


### Script 04 — Calibration Metrics And Resampling

Defines ECE, reliability buckets, bootstrap AUC intervals, and paired bootstrap AUC differences.


In [27]:
def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    """Same definition as in 03_train_models.py — bucket gap × bucket weight."""
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins[1:-1], right=False)
    ece = 0.0
    for b in range(n_bins):
        mask = idx == b
        if not mask.any():
            continue
        ece += mask.mean() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)


def reliability_points(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (mean_predicted_prob_per_bin, empirical_pos_rate_per_bin, count_per_bin)."""
    # what: build the data points behind a reliability diagram
    # how: digitise into n_bins, take per-bin mean of probs and per-bin empirical positive rate
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins[1:-1], right=False)
    mean_pred, emp_rate, counts = [], [], []
    for b in range(n_bins):
        mask = idx == b
        if mask.any():
            mean_pred.append(float(y_prob[mask].mean()))
            emp_rate.append(float(y_true[mask].mean()))
            counts.append(int(mask.sum()))
        else:
            mean_pred.append(np.nan); emp_rate.append(np.nan); counts.append(0)
    return np.array(mean_pred), np.array(emp_rate), np.array(counts)


def bootstrap_auc_ci(y_true: np.ndarray, y_prob: np.ndarray, n_iter: int = N_BOOTSTRAP,
                     seed: int = RANDOM_SEED) -> tuple[float, float, float]:
    """Resampling 95% CI on test AUC."""
    # what: standard nonparametric bootstrap; resample row indices with replacement, recompute AUC
    # why: gives a confidence interval the report can cite alongside the point estimate
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_iter, dtype=float)
    for i in range(n_iter):
        idx = rng.integers(0, n, size=n)
        # how: skip degenerate resamples (all-positive or all-negative); rare but possible
        if len(np.unique(y_true[idx])) < 2:
            aucs[i] = np.nan
            continue
        aucs[i] = roc_auc_score(y_true[idx], y_prob[idx])
    aucs = aucs[~np.isnan(aucs)]
    return float(np.mean(aucs)), float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def paired_bootstrap_auc_diff(
    y_true: np.ndarray,
    p_a: np.ndarray,
    p_b: np.ndarray,
    n_iter: int = N_BOOTSTRAP,
    seed: int = RANDOM_SEED,
) -> dict:
    """Paired bootstrap test on the AUC difference between two models.

    what: resample BOTH prediction arrays with the same indices (paired) and
          measure AUC(m_a) - AUC(m_b) on each resample.
    why:  the per-model bootstrap CIs don't tell us whether the gap between
          two models is statistically real — paired resampling accounts for
          correlation in errors across models.
    how:  p_value = 2 * min(P(diff <= 0), P(diff >= 0)), two-tailed.
          Degenerate resamples (single class) are skipped to keep the estimator
          unbiased — identical to the guard in bootstrap_auc_ci.
    """
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs: list[float] = []
    for _ in range(n_iter):
        idx = rng.integers(0, n, size=n)
        y_s = y_true[idx]
        if len(np.unique(y_s)) < 2:
            continue
        auc_a = roc_auc_score(y_s, p_a[idx])
        auc_b = roc_auc_score(y_s, p_b[idx])
        diffs.append(auc_a - auc_b)
    diffs_arr = np.array(diffs)
    mean_diff = float(np.mean(diffs_arr))
    ci_lower = float(np.percentile(diffs_arr, 2.5))
    ci_upper = float(np.percentile(diffs_arr, 97.5))
    p_val = float(2.0 * min((diffs_arr <= 0).mean(), (diffs_arr >= 0).mean()))
    return {
        "mean_diff": mean_diff,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "p_value": p_val,
    }


### Script 04 — Calibration And Reliability Plots

Fits isotonic calibrators from OOF predictions and renders per-model/combined reliability diagrams.


In [28]:
def calibrate_one(name: str, y_train: pd.Series, y_test: pd.Series,
                   models_dir: Path) -> dict | None:
    """Fit isotonic on (oof, y_train), apply to raw test, save calibrated preds."""
    # what: read OOF + raw test preds saved by 03_train_models
    model_dir = models_dir / name
    oof_path = model_dir / "preds_oof.npy"
    raw_path = model_dir / "preds_test.npz"
    if not oof_path.exists() or not raw_path.exists():
        print(f"  [{name}] missing predictions, skipping")
        return None
    oof = np.load(oof_path)
    raw = np.load(raw_path)["raw"]

    # what: fit isotonic on the train-side OOF; apply to test
    # why: monotone, distribution-free recalibration; well-suited to tree-based scores that are not probabilities
    cal_model = IsotonicRegression(out_of_bounds="clip").fit(oof, y_train.values)
    cal = cal_model.transform(raw)
    np.savez_compressed(model_dir / "preds_test_cal.npz", cal=cal.astype("float32"))
    joblib.dump(cal_model, model_dir / "isotonic.joblib")

    # what: report Brier + ECE before and after calibration
    # why: calibration should reduce Brier and ECE; if it does not, the model's ranking is good but its scores are uncalibrated
    return {
        "model": name,
        "test_auc_raw": float(roc_auc_score(y_test.values, raw)),
        "test_auc_cal": float(roc_auc_score(y_test.values, cal)),
        "brier_raw": float(brier_score_loss(y_test.values, raw)),
        "brier_cal": float(brier_score_loss(y_test.values, cal)),
        "ece_raw": expected_calibration_error(y_test.values, raw),
        "ece_cal": expected_calibration_error(y_test.values, cal),
    }


# ----------------------------------------------------------------------------
# Plots
# ----------------------------------------------------------------------------

def plot_reliability(y_true: np.ndarray, y_raw: np.ndarray, y_cal: np.ndarray,
                      name: str, out_path: Path) -> None:
    """One reliability diagram per model, raw vs calibrated vs perfect line."""
    mp_raw, er_raw, _ = reliability_points(y_true, y_raw)
    mp_cal, er_cal, _ = reliability_points(y_true, y_cal)
    fig, ax = plt.subplots(figsize=(FIG_W_HALF + 1.6, FIG_W_HALF + 1.4))
    ax.plot([0, 1], [0, 1], ls="--", lw=0.9, color=COL_DARK, label="perfect calibration")
    ax.plot(mp_raw, er_raw, "o-", color=COL_TEST, lw=1.4, label="raw")
    ax.plot(mp_cal, er_cal, "s-", color=COL_TRAIN, lw=1.4, label="isotonic-calibrated")
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("empirical positive rate")
    ax.set_title(f"Reliability diagram, {name}")
    ax.legend(loc="upper left", frameon=False)
    clean_ax(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_reliability_combined(y_true: np.ndarray, model_to_cal: dict[str, np.ndarray],
                                out_path: Path) -> None:
    """All models on one reliability diagram (calibrated only) for the report."""
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_W * 0.78))
    ax.plot([0, 1], [0, 1], ls="--", lw=0.9, color=COL_DARK, label="perfect")
    names = list(model_to_cal)
    colors = rocket_gradient(len(names))
    for color, name in zip(colors, names):
        mp, er, _ = reliability_points(y_true, model_to_cal[name])
        ax.plot(mp, er, "o-", lw=1.3, color=color, label=name)
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("empirical positive rate")
    ax.set_title("Reliability diagram, all models post calibration")
    ax.legend(fontsize=8, frameon=False, loc="upper left")
    clean_ax(ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


### Script 04 — Agreement And Per-Market Metrics

Writes the inter-model agreement matrix and per-market/per-model AUC table and heatmap.


In [29]:
def inter_model_agreement(cal_map: dict[str, np.ndarray], out_dir: Path) -> None:
    """Pearson correlation matrix of calibrated p_hat across models on test.

    A high off-diagonal means the model lineup is collapsing to one signal,
    which weakens the diversity argument behind the eight-model comparison.
    """
    if len(cal_map) < 2:
        return
    names = sorted(cal_map)
    M = np.column_stack([cal_map[n] for n in names])
    corr = np.corrcoef(M, rowvar=False)
    corr_df = pd.DataFrame(corr, index=names, columns=names)
    corr_df.round(4).to_csv(out_dir / "inter_model_agreement.csv", index_label="model")

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_W * 0.85))
    import seaborn as sns_local
    sns_local.heatmap(
        corr_df, cmap=C_MAP, vmin=0, vmax=1,
        annot=True, fmt=".2f", annot_kws={"size": 8, "color": "white"},
        square=True, linewidths=0.0, ax=ax,
        cbar_kws={"label": "Pearson r between calibrated p_hat"},
    )
    ax.set_title("Inter-model agreement on calibrated test predictions")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
    fig.tight_layout()
    fig.savefig(out_dir / "inter_model_agreement.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved inter_model_agreement.csv + .png")


def per_market_per_model_auc(cal_map: dict[str, np.ndarray],
                              y_test: np.ndarray, market_ids_test: np.ndarray,
                              out_dir: Path, min_n: int = 200) -> None:
    """Per-market test AUC for each model.

    Surfaces concentration: if one market drives the headline AUC, the others
    show weak or sub-0.5 cells. Markets with fewer than `min_n` test trades
    are skipped to avoid fold-degeneracy.
    """
    if not cal_map:
        return
    names = sorted(cal_map)
    rows = []
    for mid in pd.unique(market_ids_test):
        m = market_ids_test == mid
        if m.sum() < min_n or len(np.unique(y_test[m])) < 2:
            continue
        for name in names:
            try:
                auc = roc_auc_score(y_test[m], cal_map[name][m])
            except ValueError:
                continue
            rows.append({"market_id": str(mid), "model": name,
                         "n_test": int(m.sum()), "auc": float(auc)})
    if not rows:
        return
    df = pd.DataFrame(rows)
    pivot = df.pivot(index="model", columns="market_id", values="auc").reindex(names)
    counts = df.groupby("market_id")["n_test"].first()
    pivot.to_csv(out_dir / "per_market_per_model_auc.csv", index_label="model")

    fig, ax = plt.subplots(figsize=(FIG_W_WIDE, 0.4 * len(names) + 1.6))
    import seaborn as sns_local
    sns_local.heatmap(
        pivot, cmap=C_MAP_PERFORMANCE, vmin=0.30, vmax=0.70, center=0.5,
        annot=True, fmt=".2f", annot_kws={"size": 7},
        linewidths=0.0, ax=ax,
        cbar_kws={"label": "test AUC (calibrated)"},
    )
    ax.set_title("Per-market test AUC by model")
    xticks = [f"{m}\nn={counts[m]:,}" for m in pivot.columns]
    ax.set_xticklabels(xticks, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(pivot.index, rotation=0, fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(out_dir / "per_market_per_model_auc.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved per_market_per_model_auc.csv + .png "
          f"({pivot.shape[0]} models × {pivot.shape[1]} markets)")


### Script 04 — Permutation Importance And SHAP

Runs permutation importance for the best model and SHAP summaries for top tree-model picks when dependencies support it.


In [30]:
def permutation_importance_top_k(best_model_name: str, k: int = 15) -> pd.DataFrame | None:
    """Refit best model on full train, run sklearn permutation_importance on test, return top-k."""
    # what: identify best model from comparison.csv; refit; permute each feature on test; record AUC drop
    # why: model-agnostic feature importance (works for trees, linear, MLP) — answers "which features matter?"
    feature_cols = json.loads((OUTPUTS_DIR / "data" / "feature_cols.json").read_text())
    df = pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet")
    train = df[df["split"] == "train"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)
    X_train = train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_train = train[TARGET].astype(int)
    y_test = test[TARGET].astype(int)

    # what: re-instantiate best model with default tuned-down settings (matches 03_train_models)
    # how: import the matching factory from 03_train_models lazily to avoid duplication
    # NOTEBOOK: pca_elbow_k and make_factories are already in this notebook's
    # namespace from the Script 03 cell, so call them directly instead of
    # dynamically loading 03_train_models.py from disk.
    pca_k = pca_elbow_k(X_train)
    factories = {n: (f, s) for n, f, s in make_factories(pca_k)}
    if best_model_name not in factories:
        print(f"  best model {best_model_name} not in factories; falling back to random_forest")
        best_model_name = "random_forest"
    factory, scale = factories[best_model_name]

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_tr_arr = scaler.transform(X_train)
        X_te_arr = scaler.transform(X_test)
    else:
        scaler = None
        X_tr_arr = X_train.values
        X_te_arr = X_test.values
    clf = factory().fit(X_tr_arr, y_train)

    # what: permute each feature 5 times on test, measure AUC drop
    # why: average drop = the feature's marginal contribution; not just split-count importance
    print(f"  running permutation importance on {best_model_name} ({PERM_N_REPEATS} repeats)...")
    result = permutation_importance(clf, X_te_arr, y_test.values, n_repeats=PERM_N_REPEATS,
                                     random_state=RANDOM_SEED, n_jobs=-1, scoring="roc_auc")
    imp = pd.DataFrame({"feature": feature_cols,
                        "auc_drop_mean": result.importances_mean,
                        "auc_drop_std": result.importances_std})
    imp = imp.sort_values("auc_drop_mean", ascending=False).head(k)
    return imp


def plot_shap_summary(
    feature_names: list[str],
    mean_abs: np.ndarray,
    out_path: Path,
    top_n: int = 20,
) -> None:
    """Horizontal bar chart of mean(|SHAP value|) per feature, in report style.

    Replaces shap.summary_plot, which forces its own bright palette and
    layout and ignores the report theme. Single-series ranked bars follow
    Design.md rule 3 — rocket gradient so bar-rank reads as colour.
    """
    order = np.argsort(mean_abs)[::-1][:top_n]
    feats = [feature_names[i] for i in order]
    values = mean_abs[order]

    y_pos = np.arange(len(feats))[::-1]
    fig, ax = plt.subplots(figsize=(FIG_W, 0.32 * len(feats) + 1.0))
    bars = ax.barh(
        y_pos,
        values,
        color=rocket_gradient(len(feats))[::-1],
        edgecolor="white",
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feats)
    xmax = float(values.max()) * 1.10 if len(values) else 1.0
    ax.set_xlim(0, xmax)
    for bar, v in zip(bars, values):
        ax.text(
            v + xmax * 0.008,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.3f}",
            va="center",
            fontsize=7.5,
            color=COL_DARK,
        )
    ax.set_xlabel("mean(|SHAP value|) on top-1% picks")
    clean_ax(ax)
    fig.tight_layout(pad=1.2)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def shap_on_top_picks(best_model_name: str, top_k_pct: float = 0.01) -> None:
    """SHAP analysis restricted to the top-1% highest-confidence test predictions.

    what: compute SHAP values only for the subset of test rows where the best
          calibrated model had the highest predicted probabilities.
    why:  global permutation importance averages over all rows; SHAP on the
          top-1% picks answers "what drove the model's confidence on the bets
          it actually wanted to make?" — closer to Lecture 14 (Explainable AI)
          requirements for a report-relevant explanation.
    how:  use shap.TreeExplainer (fast, exact for tree ensembles); fall back to
          the best non-MLP model if the overall best is MLP.
          For binary classification TreeExplainer returns a list [neg, pos];
          take element [1] (positive class).
    """
    try:
        import shap  # noqa: PLC0415
    except ImportError:
        print("  [shap_on_top_picks] shap not installed — skipping (pip install shap)")
        return

    TREE_MODELS = {"decision_tree", "random_forest", "hist_gbm", "lightgbm"}

    metrics_dir = OUTPUTS_DIR / "metrics"
    models_dir = OUTPUTS_DIR / "models"

    # what: pick the right model — prefer best, but fall back if MLP
    if best_model_name not in TREE_MODELS:
        # what: scan cal_df ordering to find next-best tree model
        cal_path = metrics_dir / "calibration_summary.csv"
        if not cal_path.exists():
            print("  [shap_on_top_picks] calibration_summary.csv not found — skipping")
            return
        ordered = pd.read_csv(cal_path).sort_values("test_auc_cal", ascending=False)["model"].tolist()
        candidates = [m for m in ordered if m in TREE_MODELS]
        if not candidates:
            print("  [shap_on_top_picks] no tree-based model found — skipping")
            return
        model_name = candidates[0]
        print(f"  [shap_on_top_picks] best model is {best_model_name} (not tree); "
              f"falling back to {model_name}")
    else:
        model_name = best_model_name

    # what: reload features and data (same pipeline as permutation_importance_top_k)
    feature_cols = json.loads((OUTPUTS_DIR / "data" / "feature_cols.json").read_text())
    df = pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet")
    train = df[df["split"] == "train"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)
    X_train = train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_train = train[TARGET].astype(int)

    # NOTEBOOK: pca_elbow_k and make_factories are already in this notebook's
    # namespace from the Script 03 cell, so call them directly instead of
    # dynamically loading 03_train_models.py from disk.
    pca_k = pca_elbow_k(X_train)
    factories = {n: (f, s) for n, f, s in make_factories(pca_k)}

    if model_name not in factories:
        print(f"  [shap_on_top_picks] {model_name} not in factories — skipping")
        return
    factory, scale = factories[model_name]

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_tr_arr = scaler.transform(X_train)
        X_te_arr = scaler.transform(X_test)
    else:
        X_tr_arr = X_train.values
        X_te_arr = X_test.values

    clf = factory().fit(X_tr_arr, y_train)

    # what: select top-1% test rows by calibrated probability
    cal_npz = models_dir / model_name / "preds_test_cal.npz"
    if not cal_npz.exists():
        print(f"  [shap_on_top_picks] calibrated preds for {model_name} not found — skipping")
        return
    cal_probs = np.load(cal_npz)["cal"]
    k = max(1, int(np.ceil(top_k_pct * len(cal_probs))))
    top_idx = np.argsort(cal_probs)[-k:]
    X_top = X_te_arr[top_idx]

    print(f"  [shap_on_top_picks] running SHAP on {k} top-{top_k_pct:.0%} picks for {model_name}...")
    explainer = shap.TreeExplainer(clf)
    shap_vals = explainer.shap_values(X_top)
    # how: TreeExplainer returns several shapes depending on the model:
    #   - list [neg_class, pos_class] for older binary classifiers
    #   - 3D array (n_rows, n_features, n_classes) for newer random-forest output
    #   - 2D array (n_rows, n_features) for single-class regressors / boosters
    # Pick the positive-class slice in every case so downstream code sees 2D.
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    shap_vals = np.asarray(shap_vals)
    if shap_vals.ndim == 3:
        shap_vals = shap_vals[..., 1]

    # what: mean absolute SHAP per feature, used by both the plot and the CSV
    mean_abs = np.abs(shap_vals).mean(axis=0)
    std_abs = np.abs(shap_vals).std(axis=0)

    # what: save summary bar plot in the report theme (no shap.summary_plot)
    # why: shap.summary_plot ignores rcParams and renders bright defaults;
    #      its random-forest fallback drew an interaction beeswarm with the
    #      title overlapping the y-axis labels. The custom helper enforces
    #      rocket_gradient + clean_ax per Design.md.
    plot_path = metrics_dir / f"shap_summary_top1pct_{model_name}.png"
    plot_shap_summary(feature_cols, mean_abs, plot_path, top_n=20)
    print(f"  [shap_on_top_picks] saved {plot_path.name}")
    ranking = (
        pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs, "std_shap": std_abs})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )
    csv_path = metrics_dir / f"shap_ranking_top1pct_{model_name}.csv"
    ranking.to_csv(csv_path, index=False)
    print(f"  [shap_on_top_picks] saved {csv_path.name}")


### Script 04 — Stage Entrypoint

Discovers trained models, calibrates them, and writes all calibration/diagnostic artifacts.


In [31]:
def main_04() -> int:
    print("=" * 60)
    print("Stage 4 — Calibration, reliability, bootstrap CI, permutation importance")
    print("=" * 60)
    metrics_dir = OUTPUTS_DIR / "metrics"
    models_dir = OUTPUTS_DIR / "models"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    # what: load labels (only y_train and y_test needed for calibration loop)
    df = pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet")
    y_train = df[df["split"] == "train"][TARGET].astype(int).reset_index(drop=True)
    y_test = df[df["split"] == "test"][TARGET].astype(int).reset_index(drop=True)
    # market_ids align row-for-row with y_test, used by per-market analyses below
    market_ids_test = df[df["split"] == "test"]["market_id"].astype(str).reset_index(drop=True).values

    # what: discover which models 03_train_models actually produced predictions for
    model_names = sorted([p.name for p in models_dir.iterdir() if p.is_dir()])
    print(f"  found {len(model_names)} models with predictions: {model_names}")

    # what: calibrate every model and collect the summary rows
    cal_rows: list[dict] = []
    cal_preds_for_combined: dict[str, np.ndarray] = {}
    for name in model_names:
        row = calibrate_one(name, y_train, y_test, models_dir)
        if row is None:
            continue
        cal_rows.append(row)
        # what: per-model reliability plot
        raw = np.load(models_dir / name / "preds_test.npz")["raw"]
        cal = np.load(models_dir / name / "preds_test_cal.npz")["cal"]
        plot_reliability(y_test.values, raw, cal, name, metrics_dir / f"reliability_{name}.png")
        cal_preds_for_combined[name] = cal
        print(f"  [{name}]  AUC raw={row['test_auc_raw']:.4f} cal={row['test_auc_cal']:.4f}  "
              f"Brier raw={row['brier_raw']:.4f} -> cal={row['brier_cal']:.4f}  "
              f"ECE raw={row['ece_raw']:.4f} -> cal={row['ece_cal']:.4f}")

    # what: combined reliability figure for the report
    if cal_preds_for_combined:
        plot_reliability_combined(y_test.values, cal_preds_for_combined,
                                   metrics_dir / "reliability_combined.png")

    cal_df = pd.DataFrame(cal_rows).sort_values("test_auc_cal", ascending=False)
    cal_df.to_csv(metrics_dir / "calibration_summary.csv", index=False)

    # what: bootstrap 95% CI on test AUC for the calibrated predictions
    print("\nBootstrapping 95% CI on test AUC ...")
    ci_rows = []
    for name in model_names:
        cal_path = models_dir / name / "preds_test_cal.npz"
        if not cal_path.exists():
            continue
        cal = np.load(cal_path)["cal"]
        mean_auc, lo, hi = bootstrap_auc_ci(y_test.values, cal)
        ci_rows.append({"model": name, "test_auc_cal_mean": mean_auc,
                        "ci_lower_2_5": lo, "ci_upper_97_5": hi,
                        "ci_width": hi - lo})
        print(f"  {name:14s}  AUC = {mean_auc:.4f}  CI = [{lo:.4f}, {hi:.4f}]")
    pd.DataFrame(ci_rows).to_csv(metrics_dir / "auc_bootstrap_ci.csv", index=False)

    # what: paired bootstrap AUC differences for all model pairs
    # why:  per-model CIs don't tell us if pairwise gaps are significant; paired
    #       resampling gives a proper test and Bonferroni-corrected p-values
    print("\nPaired bootstrap AUC differences ...")
    # how:  collect calibrated preds for all models that produced a cal file
    cal_map: dict[str, np.ndarray] = {}
    for name in model_names:
        cp = models_dir / name / "preds_test_cal.npz"
        if cp.exists():
            cal_map[name] = np.load(cp)["cal"]
    pairs = [(a, b) for i, a in enumerate(sorted(cal_map)) for b in sorted(cal_map)[i + 1:]]
    n_pairs = len(pairs)
    pair_rows = []
    for m_a, m_b in pairs:
        res = paired_bootstrap_auc_diff(y_test.values, cal_map[m_a], cal_map[m_b], n_iter=200)
        bonf = min(res["p_value"] * n_pairs, 1.0)
        pair_rows.append({
            "model_a": m_a,
            "model_b": m_b,
            "mean_auc_diff": res["mean_diff"],
            "ci_lower": res["ci_lower"],
            "ci_upper": res["ci_upper"],
            "p_value": res["p_value"],
            "p_value_bonferroni": bonf,
        })
        print(f"  {m_a} vs {m_b}: diff={res['mean_diff']:+.4f} "
              f"CI=[{res['ci_lower']:+.4f},{res['ci_upper']:+.4f}] "
              f"p={res['p_value']:.3f} p_bonf={bonf:.3f}")
    if pair_rows:
        pair_df = pd.DataFrame(pair_rows)
        pair_df.to_csv(metrics_dir / "auc_pairwise.csv", index=False)
        top5 = pair_df.assign(abs_diff=pair_df["mean_auc_diff"].abs()).sort_values(
            "abs_diff", ascending=False
        ).head(5).drop(columns="abs_diff")
        print("\nTop-5 pairs by |mean AUC diff|:")
        print(top5.to_string(index=False))

    # what: permutation importance on the best model (head of the cal_df)
    if not cal_df.empty:
        best = cal_df.iloc[0]["model"]
        print(f"\nBest model by calibrated AUC: {best}")
        imp = permutation_importance_top_k(best)
        if imp is not None:
            imp.to_csv(metrics_dir / f"permutation_importance_{best}.csv", index=False)
            print(imp.to_string(index=False))

    # what: SHAP on top-1% picks for the top-2 tree-based models
    # why:  complements global permutation importance with a focused explainability
    #       view for the model's most confident predictions (Lecture 14 requirement).
    #       Running it for the top-2 tree models lets the report compare the
    #       feature drivers of the strongest gradient-boosted and bagged tree
    #       families side by side, instead of citing only the single best.
    if not cal_df.empty:
        TREE_MODELS = {"decision_tree", "random_forest", "hist_gbm", "lightgbm"}
        top_trees = [m for m in cal_df["model"].tolist() if m in TREE_MODELS][:2]
        if not top_trees:
            print("\nSHAP step: no tree-based model in the lineup, skipping.")
        for model_name in top_trees:
            print(f"\nSHAP on top-1% picks for {model_name} ...")
            shap_on_top_picks(model_name)

    # what: post-processing analyses on the multi-model prediction set
    # why:  inter-model agreement tests whether the lineup adds genuine
    #       diversity; per-market×per-model AUC surfaces concentration
    print("\nPost-processing analyses ...")
    inter_model_agreement(cal_map, metrics_dir)
    per_market_per_model_auc(cal_map, y_test.values, market_ids_test, metrics_dir)

    print(f"\nStage 4 complete. Outputs in {metrics_dir.relative_to(OUTPUTS_DIR.parent)}.")
    print("Proceed to 05_backtest.py.")
    return 0


### Script 04 — Run

Inputs: `outputs/models/<model>/preds_*` from Script 03. Outputs: calibrated predictions and diagnostics under `outputs/models/` and `outputs/metrics/`.


In [32]:
np.random.seed(RANDOM_SEED)
main_04()


Stage 4 — Calibration, reliability, bootstrap CI, permutation importance
  found 8 models with predictions: ['decision_tree', 'hist_gbm', 'lightgbm', 'logreg_l1', 'logreg_l2', 'mlp_sklearn', 'pca_logreg', 'random_forest']
  [decision_tree]  AUC raw=0.5380 cal=0.5369  Brier raw=0.2543 -> cal=0.2500  ECE raw=0.0418 -> cal=0.0202
  [hist_gbm]  AUC raw=0.5614 cal=0.5614  Brier raw=0.2465 -> cal=0.2462  ECE raw=0.0280 -> cal=0.0163
  [lightgbm]  AUC raw=0.5638 cal=0.5637  Brier raw=0.2489 -> cal=0.2471  ECE raw=0.0442 -> cal=0.0217
  [logreg_l1]  AUC raw=0.5436 cal=0.5435  Brier raw=0.2491 -> cal=0.2483  ECE raw=0.0268 -> cal=0.0174
  [logreg_l2]  AUC raw=0.5436 cal=0.5435  Brier raw=0.2491 -> cal=0.2483  ECE raw=0.0268 -> cal=0.0174
  [mlp_sklearn]  AUC raw=0.5496 cal=0.5496  Brier raw=0.2621 -> cal=0.2486  ECE raw=0.1006 -> cal=0.0332
  [pca_logreg]  AUC raw=0.5346 cal=0.5345  Brier raw=0.2495 -> cal=0.2491  ECE raw=0.0240 -> cal=0.0196
  [random_forest]  AUC raw=0.5559 cal=0.5559  Brier 

0

## Script 05 — Realistic Backtest And Headline Figure

Runs the capital-aware backtest, naive consensus baseline, strategy sensitivity grid, and headline ROI/diagnostic figures. Runtime is about 5 minutes.


### Script 05 — Imports, Constants, And Cost Inputs

Defines backtest context columns, cost/slippage assumptions, and event timestamps.


In [33]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

apply_theme()

TARGET = "bet_correct"

BACKTEST_CONTEXT = DATA_DIR / "backtest_context.parquet"
BACKTEST_CONTEXT_REQUIRED_COLS = {
    "split", "row_in_split", "market_id", "timestamp", "usd_amount",
    "price", "token_amount", "pre_yes_price_corrected",
    "taker", "taker_direction", "nonusdc_side",
}

# what: realism constants (cost floor caps payoff at 19x; gas + slippage applied per trade)
COST_FLOOR = 0.05
GAS_COST_USD = 0.50
SLIPPAGE_THRESHOLD = 0.25
SLIPPAGE_FACTOR = 0.05

# what: real-world events that bracket the test cohort; bets resolve at the event
STRIKE_EVENT_UTC = pd.Timestamp("2026-02-28T06:35:00", tz="UTC").timestamp()
CEASEFIRE_EVENT_UTC = pd.Timestamp("2026-04-07T23:59:59", tz="UTC").timestamp()


### Script 05 — Cost, Edge, And Strategy Masks

Converts trade side into market-implied side probability, computes model edge, and defines strategy filters.


In [34]:
def trader_side_is_yes(test: pd.DataFrame) -> np.ndarray:
    """Return 1 if the trade is economically exposed to YES, else 0 for NO."""
    side_buy = test["side_buy"].values
    outcome_yes = test["outcome_yes"].values
    return side_buy * outcome_yes + (1 - side_buy) * (1 - outcome_yes)


def market_side_probability(test: pd.DataFrame) -> np.ndarray:
    """Market-implied probability of the side the trader is economically taking."""
    if "pre_yes_price_corrected" not in test.columns:
        raise ValueError(
            "pre_yes_price_corrected is required. Run 01_data_prep.py with "
            "submission/data/backtest_context.parquet present."
        )
    p_yes = test["pre_yes_price_corrected"].values
    side_yes = trader_side_is_yes(test)
    return np.where(side_yes == 1, p_yes, 1 - p_yes)


def compute_cost_and_edge(test: pd.DataFrame, p_hat: np.ndarray, cost_floor: float = COST_FLOOR):
    """Per-trade cost paid for the winning side + model edge (p_hat - cost)."""
    # what: every trade has a buy/sell side and a yes/no outcome; cost is the price for the winning side
    # how: BUY YES at p_yes -> cost = p_yes; BUY NO at (1-p_yes) -> cost = 1 - p_yes; etc
    # why: edge = model's view of P(win) minus market's implied price = expected profit per $1 staked
    cost = market_side_probability(test)
    cost = np.clip(cost, cost_floor, 1.0 - cost_floor)
    edge = p_hat - cost
    return cost, edge


# ----------------------------------------------------------------------------
# Strategy masks
# ----------------------------------------------------------------------------

def strategy_masks(p_hat: np.ndarray, edge: np.ndarray, cost: np.ndarray,
                    time_to_deadline: np.ndarray) -> dict[str, np.ndarray]:
    """Return a dict {strategy_name: boolean mask of which trades to take}."""
    n = len(p_hat)

    def top_k(scores: np.ndarray, k_pct: float) -> np.ndarray:
        # what: boolean mask selecting the top k_pct by score
        k = max(1, int(n * k_pct))
        m = np.zeros(n, dtype=bool)
        m[np.argsort(scores)[-k:]] = True
        return m

    return {
        # what: high-confidence picks (model says >= 99% sure to win)
        "phat_gt_0.99":   p_hat >= 0.99,
        "phat_gt_0.95":   p_hat >= 0.95,
        "phat_gt_0.9":    p_hat >= 0.90,
        # what: top-1% by phat (compares across models even when phat distributions differ)
        "top1pct_phat":   top_k(p_hat, 0.01),
        # what: top-K% by edge (model's predicted alpha)
        "top1pct_edge":   top_k(edge, 0.01),
        "top5pct_edge":   top_k(edge, 0.05),
        # what: general EV rule — bet whenever edge > 2 cents
        "general_ev":     edge > 0.02,
        # what: late + edge filter — focuses on insider-style opportunities near deadline
        "general_ev_late": (edge > 0.02) & (time_to_deadline < 6 * 3600),
        # what: cheap + edge — bets where the market underprices an underdog
        "general_ev_cheap": (edge > 0.02) & (cost < 0.30),
        # what: high-edge late-cheap "home run" combo (asymmetric-info hypothesis)
        "home_run":       (edge > 0.20) & (cost < 0.30) & (time_to_deadline < 6 * 3600),
    }


### Script 05 — Capital-Aware Execution Loop

Simulates bankroll, concentration, volume caps, gas, slippage, settlement timing, and per-trade PnL.


In [35]:
def realistic_backtest(signal_mask: np.ndarray, cost: np.ndarray, bet_correct: np.ndarray,
                        timestamps: np.ndarray, market_ids: np.ndarray, usd_amount: np.ndarray,
                        market_res_times: dict, *,
                        initial_capital: float = 10_000, max_bet_usd: float = 100.0,
                        max_bet_pct_capital: float = 0.05, max_bet_pct_volume: float = 0.10,
                        max_concentration_pct: float = 0.20, gas_cost: float = GAS_COST_USD,
                        slippage_threshold: float = SLIPPAGE_THRESHOLD,
                        slippage_factor: float = SLIPPAGE_FACTOR,
                        liquidity_scaler: float = 1.0) -> dict:
    """Chronologically execute every signal subject to capital, concentration, gas, slippage."""
    # what: select only the signal trades and order them by timestamp
    sig_idx = np.where(signal_mask)[0]
    if len(sig_idx) == 0:
        return {"n_signals": 0, "n_executed": 0, "final_capital": initial_capital,
                "roi": 0.0, "max_drawdown": 0.0}
    order = sig_idx[np.argsort(timestamps[sig_idx])]

    # what: runtime state
    capital = initial_capital
    open_positions: dict[str, float] = {}     # market_id -> $ tied up
    pending: dict[str, list] = {}             # market_id -> [(resolve_ts, return_amt, entry_bet)]
    max_capital, min_capital = capital, capital
    n_executed, skipped = 0, 0

    def release_resolved(now_ts: int) -> None:
        # what: any market that has now resolved returns its winnings/zeroes to capital
        nonlocal capital, max_capital, min_capital
        for mid in list(pending.keys()):
            still = []
            for res_ts, ret_amt, entry_bet in pending[mid]:
                if res_ts <= now_ts:
                    capital += ret_amt
                    open_positions[mid] = open_positions.get(mid, 0) - entry_bet
                    if open_positions[mid] <= 0.01:
                        open_positions.pop(mid, None)
                else:
                    still.append((res_ts, ret_amt, entry_bet))
            if still:
                pending[mid] = still
            else:
                pending.pop(mid, None)
        max_capital = max(max_capital, capital)
        min_capital = min(min_capital, capital)

    for i in order:
        ts = int(timestamps[i])
        release_resolved(ts)

        # what: refuse to bet if broke
        if capital <= 1.0:
            skipped += 1
            continue

        # what: bet sizing combines bankroll-pct, absolute max, and a fraction of the trade's volume
        # why: bankroll-pct keeps drawdown bounded; vol-cap keeps us from being too large to fill
        effective_trade_usd = float(usd_amount[i]) * liquidity_scaler
        bet = min(max_bet_usd, max_bet_pct_capital * capital,
                  max_bet_pct_volume * effective_trade_usd)

        # what: concentration cap — never have more than X% of bankroll riding on one market
        mid = str(market_ids[i])
        in_market = open_positions.get(mid, 0)
        room = max_concentration_pct * capital - in_market
        if room < 1.0:
            skipped += 1
            continue
        bet = min(bet, room)
        if bet < 1.0:
            skipped += 1
            continue

        # what: slippage — large bets relative to trade size pay an effective cost penalty
        eff_cost = cost[i]
        if bet > slippage_threshold * effective_trade_usd:
            eff_cost = min(0.99, eff_cost * (1 + slippage_factor))

        # what: commit capital, schedule resolution
        capital -= bet
        open_positions[mid] = open_positions.get(mid, 0) + bet
        # what: gas debited up-front so it shows in min_capital tracking
        capital -= gas_cost

        if bet_correct[i]:
            # what: payoff = stake / cost (gross). On win we add it back at resolution time
            payoff = bet / eff_cost
            res_ts = market_res_times.get(mid, ts + 86400)
            pending.setdefault(mid, []).append((res_ts, payoff, bet))
        else:
            # what: loss — capital is gone, stake stays "tied up" until resolution to keep concentration accurate
            res_ts = market_res_times.get(mid, ts + 86400)
            pending.setdefault(mid, []).append((res_ts, 0.0, bet))

        n_executed += 1
        max_capital = max(max_capital, capital)
        min_capital = min(min_capital, capital)

    # what: settle anything still pending at the end of the test cohort
    release_resolved(int(max(timestamps[order])) + 86400 * 365)

    drawdown = (max_capital - min_capital) if max_capital > 0 else 0.0
    return {"n_signals": int(signal_mask.sum()), "n_executed": int(n_executed),
            "final_capital": float(capital), "roi": float(capital / initial_capital - 1),
            "max_drawdown": float(drawdown)}


### Script 05 — Naive Baseline And Context Attachment

Builds the consensus baseline, attaches backtest-only context, and defines helper metrics.


In [36]:
def naive_consensus_phat(test: pd.DataFrame) -> np.ndarray:
    """Naive baseline: use the market-implied probability of the trader's side."""
    # what: a free heuristic that requires no model — just use the market's own price
    # why: every model must beat THIS to claim it adds value (Stage B1b falsification)
    # what: baseline prob of winning = market's price for the trader's chosen side
    return market_side_probability(test)


# ----------------------------------------------------------------------------
# Backtest context and diagnostics
# ----------------------------------------------------------------------------

def attach_backtest_context(test: pd.DataFrame) -> pd.DataFrame:
    """Attach raw backtest-only fields and fail fast if they are missing or misaligned."""
    if not BACKTEST_CONTEXT.exists():
        raise SystemExit(
            f"Missing {BACKTEST_CONTEXT}. The realistic backtest requires the "
            "bundled backtest context for corrected YES prices and trade USD."
        )
    ctx = pd.read_parquet(BACKTEST_CONTEXT)
    missing = sorted(BACKTEST_CONTEXT_REQUIRED_COLS - set(ctx.columns))
    if missing:
        raise SystemExit(f"{BACKTEST_CONTEXT.name} is missing required columns: {missing}")

    ctx_test = ctx[ctx["split"] == "test"].reset_index(drop=True)
    if len(ctx_test) != len(test):
        raise SystemExit(
            f"backtest context row mismatch: test={len(test):,}, "
            f"context={len(ctx_test):,}"
        )
    expected_row = np.arange(len(test))
    if not np.array_equal(ctx_test["row_in_split"].to_numpy(), expected_row):
        raise SystemExit("backtest context row_in_split is not aligned to test row order")
    key_ok = (
        test["market_id"].astype(str).to_numpy() == ctx_test["market_id"].astype(str).to_numpy()
    ).all() and (
        test["timestamp"].to_numpy() == ctx_test["timestamp"].to_numpy()
    ).all()
    if not key_ok:
        raise SystemExit("backtest context market_id/timestamp keys do not align to test rows")
    if not ctx_test["pre_yes_price_corrected"].between(0, 1).all():
        raise SystemExit("pre_yes_price_corrected must be in [0, 1]")
    if not (np.isfinite(ctx_test["usd_amount"]).all() and (ctx_test["usd_amount"] >= 0).all()):
        raise SystemExit("usd_amount must be finite and non-negative for every backtest row")

    out = test.copy()
    for col in [
        "usd_amount", "price", "token_amount", "pre_yes_price_corrected",
        "taker", "taker_direction", "nonusdc_side",
    ]:
        out[col] = ctx_test[col].values
    print(
        "  attached backtest context: "
        f"pre_yes mean={out['pre_yes_price_corrected'].mean():.3f}, "
        f"usd mean=${out['usd_amount'].mean():,.2f}"
    )
    return out


def safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:
    """AUC helper that returns NaN for degenerate scores or labels."""
    if len(np.unique(y_true)) < 2 or np.nanstd(score) == 0:
        return float("nan")
    return float(roc_auc_score(y_true, score))


### Script 05 — Diagnostics

Writes residual-edge, consensus, and SELL-semantics checks used to validate the backtest story.


In [37]:
def residualize(x: np.ndarray, control: np.ndarray) -> np.ndarray:
    """Linear residual of x after projecting on one control variable."""
    if np.nanstd(control) == 0:
        return x - np.nanmean(x)
    a, b = np.polyfit(control, x, 1)
    return x - (a * control + b)


def write_residual_edge_diagnostics(test: pd.DataFrame, model_preds: dict[str, np.ndarray],
                                    out_dir: Path) -> None:
    """Does p_hat add signal beyond the market-implied probability of the same side?"""
    y = test[TARGET].astype(int).values
    market_prob = market_side_probability(test)
    rows = []
    for model_name, p_hat in model_preds.items():
        edge = p_hat - market_prob
        edge_resid = residualize(edge, market_prob)
        y_resid = residualize(y.astype(float), market_prob)
        partial_corr = float(np.corrcoef(edge_resid, y_resid)[0, 1])
        rows.append({
            "model": model_name,
            "auc_p_hat": safe_auc(y, p_hat),
            "auc_market_prob": safe_auc(y, market_prob),
            "auc_edge": safe_auc(y, edge),
            "residual_edge_auc": safe_auc(y, edge_resid),
            "partial_corr_edge_y_given_market_prob": partial_corr,
            "mean_edge": float(np.mean(edge)),
            "share_positive_edge": float(np.mean(edge > 0)),
            "n": int(len(y)),
        })
    pd.DataFrame(rows).to_csv(out_dir / "diagnostics_residual_edge.csv", index=False)


def write_consensus_diagnostics(test: pd.DataFrame, model_preds: dict[str, np.ndarray],
                                out_dir: Path) -> None:
    """Top-pick decomposition: consensus-following vs contrarian selections."""
    y = test[TARGET].astype(int).values
    pre_yes = test["pre_yes_price_corrected"].values
    side_yes = trader_side_is_yes(test)
    consensus_yes = pre_yes >= 0.5
    with_consensus = side_yes == consensus_yes.astype(int)
    market_prob = market_side_probability(test)
    n = len(test)

    rows = []
    for model_name, p_hat in model_preds.items():
        edge = p_hat - market_prob
        selectors = {
            "top1pct_phat": p_hat,
            "top1pct_edge": edge,
        }
        for selector, score in selectors.items():
            k = max(1, int(n * 0.01))
            idx = np.argsort(score)[-k:]
            with_idx = idx[with_consensus[idx]]
            against_idx = idx[~with_consensus[idx]]
            market_counts = test.iloc[idx]["market_id"].value_counts()
            rows.append({
                "model": model_name,
                "selector": selector,
                "n_picks": int(k),
                "hit_rate": float(y[idx].mean()),
                "pct_with_consensus": float(with_consensus[idx].mean()),
                "pct_against_consensus": float((~with_consensus[idx]).mean()),
                "with_consensus_hit_rate": float(y[with_idx].mean()) if len(with_idx) else np.nan,
                "against_consensus_hit_rate": float(y[against_idx].mean()) if len(against_idx) else np.nan,
                "pre_yes_mean": float(pre_yes[idx].mean()),
                "pre_yes_median": float(np.median(pre_yes[idx])),
                "n_unique_markets": int(market_counts.size),
                "top_market_share": float(market_counts.iloc[0] / k) if len(market_counts) else 0.0,
            })
    pd.DataFrame(rows).to_csv(out_dir / "diagnostics_consensus.csv", index=False)


def write_sell_semantics_diagnostics(test: pd.DataFrame, model_preds: dict[str, np.ndarray],
                                     out_dir: Path) -> None:
    """Check how often SELL rows look like closing trades rather than fresh directional bets."""
    raw = test[["market_id", "timestamp", "taker", "taker_direction", "nonusdc_side", TARGET]].copy()
    raw["market_id"] = raw["market_id"].astype(str)
    raw["taker"] = raw["taker"].astype(str)
    raw["nonusdc_side"] = raw["nonusdc_side"].astype(str)
    raw["is_sell"] = raw["taker_direction"].astype(str).str.upper().eq("SELL")
    raw["is_buy"] = raw["taker_direction"].astype(str).str.upper().eq("BUY")

    grp = raw.groupby(["market_id", "taker", "nonusdc_side"], sort=False)
    raw["prior_buys_same_side"] = grp["is_buy"].cumsum().shift(1).fillna(0)
    raw.loc[grp.head(1).index, "prior_buys_same_side"] = 0

    sells = raw[raw["is_sell"]]
    n_sells = int(len(sells))
    n_closing = int((sells["prior_buys_same_side"] >= 1).sum())
    detail: dict[str, object] = {
        "n_test_rows": int(len(raw)),
        "n_sells": n_sells,
        "n_sells_closing": n_closing,
        "n_sells_open_short": int(n_sells - n_closing),
        "pct_sells_closing": float(n_closing / max(n_sells, 1)),
    }

    y = raw[TARGET].astype(int).values
    ml_preds = {k: v for k, v in model_preds.items() if k != "naive_consensus"}
    if ml_preds:
        best_name = max(ml_preds, key=lambda k: safe_auc(y, ml_preds[k]))
        p_hat = ml_preds[best_name]
        k = max(1, int(len(raw) * 0.01))
        top_idx = np.argsort(p_hat)[-k:]
        top = raw.iloc[top_idx]
        top_sells = top[top["is_sell"]]
        detail.update({
            "top1pct_model": best_name,
            "top1pct_n": int(k),
            "top1pct_sell_share": float(len(top_sells) / k),
            "top1pct_sells_closing_share": float(
                (top_sells["prior_buys_same_side"] >= 1).mean()
            ) if len(top_sells) else None,
        })
    (out_dir / "diagnostics_sell_semantics.json").write_text(json.dumps(detail, indent=2))


### Script 05 — Backtest Figures

Renders the headline overview, per-market PnL breakdown, and edge-distribution chart.


In [38]:
def render_overview(df: pd.DataFrame, out_path: Path,
                     scenario: dict = {"initial_capital": 10000, "max_bet_pct": 0.05, "liquidity_scaler": 1.0}) -> None:
    """Build the heatmap of ROI per (strategy, model) for the headline scenario."""
    # what: filter to one scenario row per (model, strategy)
    sub = df.copy()
    for k, v in scenario.items():
        sub = sub[sub[k] == v]
    if sub.empty:
        print("  no rows for headline scenario; skipping overview chart")
        return
    # Layout: rows = models (y-axis), columns = strategies (x-axis).
    pivot = sub.pivot_table(index="model", columns="strategy", values="roi", aggfunc="first")
    counts = sub.pivot_table(index="model", columns="strategy", values="n_executed",
                              aggfunc="first").reindex_like(pivot).fillna(0).astype(int)
    # what: order rows/cols so report figure is consistent run-to-run
    strategy_order = ["phat_gt_0.99", "phat_gt_0.95", "phat_gt_0.9", "top1pct_phat",
                      "top1pct_edge", "top5pct_edge", "general_ev", "general_ev_late",
                      "general_ev_cheap", "home_run"]
    model_order = ["hist_gbm", "lightgbm", "random_forest", "decision_tree",
                   "logreg_l1", "logreg_l2", "pca_logreg", "mlp_sklearn",
                   "naive_consensus"]
    pivot = pivot.reindex(
        index=[m for m in model_order if m in pivot.index],
        columns=[s for s in strategy_order if s in pivot.columns],
    )
    counts = counts.reindex_like(pivot).fillna(0).astype(int)

    norm = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=0.30)

    fig, ax = plt.subplots(figsize=(FIG_W_WIDE + 0.8, 0.42 * len(pivot.index) + 1.6))
    img = ax.imshow(pivot.values, cmap=C_MAP_PERFORMANCE, norm=norm, aspect="auto")
    for i, _ in enumerate(pivot.index):
        for j, _ in enumerate(pivot.columns):
            val = pivot.values[i, j]
            n = counts.values[i, j]
            if pd.isna(val) or n == 0:
                ax.text(j, i, "—", ha="center", va="center", color="#999", fontsize=8)
                continue
            color = "white" if abs(val) > 0.30 else COL_DARK
            ax.text(j, i, f"{val * 100:+.0f}%", ha="center", va="center",
                    color=color, fontsize=10, fontweight="bold")
            ax.text(j, i + 0.32, f"n={n:,}", ha="center", va="center",
                    color=color, fontsize=7, alpha=0.85)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=9, rotation=35, ha="left",
                       rotation_mode="anchor")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.tick_params(axis="x", labeltop=True, labelbottom=False, top=True, bottom=False)
    ax.set_xlabel("")
    ax.set_ylabel("")
    cbar = fig.colorbar(img, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("ROI on $10K bankroll", fontsize=9)
    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


# ----------------------------------------------------------------------------
# Diagnostics: per-market PnL + edge-bucket hit rate (added per robustness review)
# ----------------------------------------------------------------------------


def per_market_pnl_breakdown(model_name: str, p_hat: np.ndarray, edge: np.ndarray,
                              cost: np.ndarray, bet_correct: np.ndarray,
                              market_ids: np.ndarray, mask: np.ndarray,
                              out_path: Path) -> pd.DataFrame:
    """Per-market PnL breakdown for a single (model, strategy) pair.

    Tells the reviewer whether headline ROI comes from one market or many.
    With only 10 test markets, concentration is a credibility question.
    """
    sel = mask & (~np.isnan(p_hat))
    if sel.sum() == 0:
        return pd.DataFrame()
    rows = []
    for mid in np.unique(market_ids[sel]):
        m = sel & (market_ids == mid)
        n = int(m.sum())
        if n == 0:
            continue
        # Simplified PnL: stake $1 per signal, win pays 1/cost, lose pays -1
        wins = bet_correct[m].astype(int)
        c = cost[m]
        pnl = np.where(wins == 1, (1.0 / c) - 1.0, -1.0)
        rows.append({"market_id": str(mid), "n_signals": n,
                     "hit_rate": float(wins.mean()),
                     "total_pnl": float(pnl.sum()),
                     "pnl_per_signal": float(pnl.mean())})
    df = pd.DataFrame(rows).sort_values("total_pnl", ascending=False)
    df.to_csv(out_path, index=False)
    return df


def plot_edge_distribution(edge: np.ndarray, bet_correct: np.ndarray,
                            mask_top_k: np.ndarray, out_path: Path) -> None:
    """Edge distribution + hit rate per edge bucket, direct test of edge as signal.

    Left panel uses the rocket two-anchor pair (all trades vs top-1%) since the
    contrast there is two cohorts, not good vs bad. Right panel uses the
    performance red/green because it carries an above/below 0.5 value judgement.
    """
    fig, axes = plt.subplots(1, 2, figsize=(FIG_W_WIDE, FIG_W_WIDE * 0.42))
    axes[0].hist(edge, bins=80, alpha=0.55, label="all test trades", color=COL_BAR)
    if mask_top_k.any():
        axes[0].hist(edge[mask_top_k], bins=80, alpha=0.75,
                     label="top-1% by p_hat", color=COL_BAR_ALT)
    axes[0].axvline(0, color=COL_DARK, lw=0.6)
    axes[0].axvline(0.02, color=COL_DARK, lw=0.5, ls="--",
                    label="general +EV threshold (0.02)")
    axes[0].axvline(0.20, color=COL_DARK, lw=0.5, ls=":",
                    label="home-run threshold (0.20)")
    axes[0].set_xlabel("edge = p_hat − cost")
    axes[0].set_ylabel("trade count")
    axes[0].set_title("Edge distribution")
    axes[0].legend(fontsize=8, frameon=False)
    clean_ax(axes[0])

    bucket_edges = [-1, -0.2, -0.05, 0, 0.02, 0.05, 0.10, 0.20, 0.50, 1]
    labels = [f"{a:.2f}-{b:.2f}" for a, b in zip(bucket_edges[:-1], bucket_edges[1:])]
    bucket = np.digitize(edge, bucket_edges[1:-1], right=False)
    rates, counts = [], []
    for b in range(len(labels)):
        m = bucket == b
        rates.append(bet_correct[m].mean() if m.any() else 0)
        counts.append(int(m.sum()))
    bars = axes[1].bar(
        labels, rates,
        color=[COL_PERF_BAD if r < 0.5 else COL_PERF_GOOD for r in rates],
    )
    axes[1].axhline(0.5, color=COL_DARK, ls="--", lw=0.6)
    axes[1].set_xlabel("edge bucket")
    axes[1].set_ylabel("empirical hit rate")
    axes[1].set_title("Hit rate by edge bucket")
    for bar, c in zip(bars, counts):
        axes[1].text(bar.get_x() + bar.get_width() / 2, 0.02, f"n={c}",
                     ha="center", fontsize=7, rotation=90, color="white")
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")
    clean_ax(axes[1])
    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


### Script 05 — Stage Entrypoint

Loads calibrated predictions, evaluates strategy sensitivity, runs diagnostics, and writes the headline figure.


In [39]:
def main_05() -> int:
    print("=" * 60)
    print("Stage 5 — Realistic backtest, naive baseline, overview chart")
    print("=" * 60)
    out_dir = OUTPUTS_DIR / "backtest"
    out_dir.mkdir(parents=True, exist_ok=True)

    # what: load test rows + auxiliary columns
    print("  loading test data and predictions ...")
    df = pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet")
    test = df[df["split"] == "test"].reset_index(drop=True).copy()
    test["market_id"] = test["market_id"].astype(str)
    test = attach_backtest_context(test)

    # what: per-market resolve_ts, reconstructed from log_time_to_deadline_hours, capped at the ceasefire event
    # why: prior code assumed all 10 test markets resolve at the ceasefire. In reality the recovered deadlines
    #      span 28 days (Mar 19 to Dec 31 2026). Markets with deadlines before the ceasefire resolve at their
    #      own deadline; markets with later deadlines resolve at the ceasefire event itself. A single
    #      cohort-level resolve_ts held capital tied up 7 to 90 days too long, biasing ROI and concentration.
    # how: deadline_ts = timestamp + exp(log_time_to_deadline_hours) * 3600; median over the market's trades.
    test["recovered_deadline"] = (
        test["timestamp"] + np.exp(test["log_time_to_deadline_hours"]) * 3600
    )
    per_market_resolve = (
        test.groupby("market_id")["recovered_deadline"].median()
        .fillna(CEASEFIRE_EVENT_UTC)
        .clip(upper=CEASEFIRE_EVENT_UTC)
    )
    test["time_to_deadline"] = (
        test["market_id"].map(per_market_resolve).values
        - test["timestamp"].astype(float).values
    )
    market_res_times = {str(mid): int(ts) for mid, ts in per_market_resolve.items()}

    # what: gather calibrated predictions written by 04_calibration.py
    models_dir = OUTPUTS_DIR / "models"
    model_preds = {}
    for d in sorted(models_dir.iterdir()):
        cal_path = d / "preds_test_cal.npz"
        if cal_path.exists():
            model_preds[d.name] = np.load(cal_path)["cal"]
    # what: also include the naive consensus baseline as an extra "model"
    model_preds["naive_consensus"] = naive_consensus_phat(test)
    print(f"  models in this run: {list(model_preds)}")

    # what: write compact claim diagnostics before the trading grid
    # why: the report needs to distinguish residual model signal from consensus-following
    write_residual_edge_diagnostics(test, model_preds, out_dir)
    write_consensus_diagnostics(test, model_preds, out_dir)
    write_sell_semantics_diagnostics(test, model_preds, out_dir)
    print("  wrote residual-edge, consensus, and SELL-semantics diagnostics")

    # what: parameter grid for the sensitivity sweep
    capitals = [1_000, 10_000, 100_000]
    bet_pcts = [0.01, 0.05, 0.10]
    liquidity_scalers = [1.0, 0.10]   # 1.0 = no copycats; 0.10 = 10x copycats sharing fill

    rows: list[dict] = []
    bet_correct = test[TARGET].astype(int).values
    timestamps = test["timestamp"].astype(float).values
    market_ids = test["market_id"].values
    usd_amount = test["usd_amount"].astype(float).values
    time_to_deadline = test["time_to_deadline"].values

    # what: outer loop = model, inner = strategy x scenario grid
    for model_name, p_hat in model_preds.items():
        cost, edge = compute_cost_and_edge(test, p_hat)
        masks = strategy_masks(p_hat, edge, cost, time_to_deadline)
        for strat_name, mask in masks.items():
            for capital in capitals:
                for bet_pct in bet_pcts:
                    for ls in liquidity_scalers:
                        out = realistic_backtest(mask, cost, bet_correct, timestamps,
                                                 market_ids, usd_amount, market_res_times,
                                                 initial_capital=capital,
                                                 max_bet_pct_capital=bet_pct,
                                                 liquidity_scaler=ls)
                        rows.append({"model": model_name, "strategy": strat_name,
                                     "initial_capital": capital, "max_bet_pct": bet_pct,
                                     "liquidity_scaler": ls, **out})
            print(f"  {model_name} / {strat_name}: signals={int(mask.sum()):,}")

    sens = pd.DataFrame(rows)
    sens.to_csv(out_dir / "sensitivity.csv", index=False)
    print(f"  wrote {len(sens)} sensitivity rows -> sensitivity.csv")

    # what: headline overview chart for the report main body
    render_overview(sens, out_dir / "overview.png")
    print(f"  wrote overview.png")

    # what: falsification check — for each strategy in the headline scenario, does the best model beat naive?
    headline = sens[(sens["initial_capital"] == 10_000) & (sens["max_bet_pct"] == 0.05) &
                    (sens["liquidity_scaler"] == 1.0)]
    falsification: dict = {}
    for strat in headline["strategy"].unique():
        sub = headline[headline["strategy"] == strat]
        naive_roi = float(sub[sub["model"] == "naive_consensus"]["roi"].iloc[0]) \
            if (sub["model"] == "naive_consensus").any() else None
        if naive_roi is None:
            continue
        ml = sub[sub["model"] != "naive_consensus"].sort_values("roi", ascending=False)
        if ml.empty:
            continue
        best = ml.iloc[0]
        falsification[strat] = {"naive_roi": naive_roi, "best_model": best["model"],
                                 "best_model_roi": float(best["roi"]),
                                 "ml_beats_naive": bool(best["roi"] > naive_roi)}
    (out_dir / "falsification.json").write_text(json.dumps(falsification, indent=2))
    print("\nFalsification (does the best ML model beat the naive market-favorite baseline?):")
    for s, r in falsification.items():
        verdict = "yes" if r["ml_beats_naive"] else "no"
        print(f"  {s:20s}  naive={r['naive_roi']*100:+.1f}%  "
              f"best_ml={r['best_model']}={r['best_model_roi']*100:+.1f}%  -> {verdict}")

    # === Diagnostics added for robustness: per-market PnL + edge buckets ===
    # what: pick the best ML model from the headline scenario for diagnostics
    # why: per-market PnL shows whether headline ROI is concentrated; edge buckets
    #      directly test "higher predicted edge = more wins" for the report.
    print("\nDiagnostics: per-market PnL + edge-bucket hit rate ...")
    headline = sens[(sens["initial_capital"] == 10_000) &
                    (sens["max_bet_pct"] == 0.05) &
                    (sens["liquidity_scaler"] == 1.0)]
    ml_only = headline[headline["model"] != "naive_consensus"].sort_values(
        "roi", ascending=False)
    if not ml_only.empty:
        best_model = ml_only.iloc[0]["model"]
        print(f"  best model in headline: {best_model}")
        p_hat_best = model_preds[best_model]
        cost_b, edge_b = compute_cost_and_edge(test, p_hat_best)
        masks_b = strategy_masks(p_hat_best, edge_b, cost_b, time_to_deadline)
        # Per-market PnL on general_ev (the broadest strategy)
        pm = per_market_pnl_breakdown(best_model, p_hat_best, edge_b, cost_b,
                                       bet_correct, market_ids,
                                       masks_b["general_ev"],
                                       out_dir / f"per_market_pnl_{best_model}.csv")
        if not pm.empty:
            print(f"  per-market PnL ({best_model} on general_ev):")
            print(pm.to_string(index=False))
        # Edge distribution + hit rate
        plot_edge_distribution(edge_b, bet_correct, masks_b["top1pct_phat"],
                                out_dir / f"edge_distribution_{best_model}.png")
        print(f"  wrote edge_distribution_{best_model}.png")

    print(f"\nStage 5 complete. Outputs in {out_dir.relative_to(OUTPUTS_DIR.parent)}.")
    print("Proceed to 06_tuning_optuna.py.")
    return 0


### Script 05 — Run

Inputs: calibrated predictions from Script 04 plus `data/backtest_context.parquet`. Outputs: backtest tables, diagnostics, and figures under `outputs/backtest/`.


In [40]:
np.random.seed(RANDOM_SEED)
main_05()


Stage 5 — Realistic backtest, naive baseline, overview chart
  loading test data and predictions ...
  attached backtest context: pre_yes mean=0.262, usd mean=$156.93
  models in this run: ['decision_tree', 'hist_gbm', 'lightgbm', 'logreg_l1', 'logreg_l2', 'mlp_sklearn', 'pca_logreg', 'random_forest', 'naive_consensus']
  wrote residual-edge, consensus, and SELL-semantics diagnostics
  decision_tree / phat_gt_0.99: signals=0
  decision_tree / phat_gt_0.95: signals=747
  decision_tree / phat_gt_0.9: signals=3,351
  decision_tree / top1pct_phat: signals=2,571
  decision_tree / top1pct_edge: signals=2,571
  decision_tree / top5pct_edge: signals=12,858
  decision_tree / general_ev: signals=126,268
  decision_tree / general_ev_late: signals=338
  decision_tree / general_ev_cheap: signals=79,495
  decision_tree / home_run: signals=264
  hist_gbm / phat_gt_0.99: signals=129
  hist_gbm / phat_gt_0.95: signals=1,027
  hist_gbm / phat_gt_0.9: signals=2,160
  hist_gbm / top1pct_phat: signals=2,57

0

## Script 06 — Optuna Tuning (Optional)

Optional hyperparameter tuning for Random Forest, HistGBM, MLP, and LightGBM. All tuning cells are gated off by default because they are expensive.


### Script 06 — Imports And Optional Dependencies

Loads Optuna/LightGBM when available and records availability flags for gated tuning cells.


In [41]:
import argparse
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import (HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# what: optuna is required for this stage; if missing, mark unavailable
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    print("optuna not installed; this cell will no-op unless you install it.")
    OPTUNA_AVAILABLE = False

# what: LightGBM is a soft dependency. If it is not installed in py312 the
# tuning loop's plan defers it gracefully rather than crashing.
try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    lgb = None
    LIGHTGBM_AVAILABLE = False


TARGET = "bet_correct"


### Script 06 — Full Search Spaces

Canonical tuning spaces for Random Forest, HistGBM, MLP, and LightGBM.


In [42]:
def rf_search_space(trial) -> dict:
    """Random Forest space."""
    # what: bound the depth and leaf-size to keep per-trial fit time under ~5 min
    # why: an unbounded RF (max_depth=None) on 1.1M rows took 45 min/trial -> infeasible overnight
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "max_depth": trial.suggest_categorical("max_depth", [6, 8, 10, 12, 15]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 100, 1000, log=True),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3]),
    }


def hgbm_search_space(trial) -> dict:
    """HistGradientBoosting space."""
    # what: standard HGBM tunables; max_iter capped at 500 for wall-time
    return {
        "max_iter": trial.suggest_int("max_iter", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_categorical("max_depth", [4, 6, 8, 10, "none"]),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 127, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 50, 500, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 1.0, log=True),
    }


def mlp_search_space(trial) -> dict:
    """sklearn MLP space — picks one of 6 architectures plus the usual reg/lr knobs."""
    # what: choose the architecture from a discrete set rather than independently per-layer
    # why: independent per-layer search blows up the space; preset arches converge faster
    architectures = [(64,), (128,), (64, 32), (128, 64), (128, 64, 32), (256, 128, 64)]
    # Stash the arch list on the trial so the refit block reads the same
    # mapping the study was searched against, even if the canonical space and
    # a Tier-1 space disagree on the high-index architectures.
    trial.set_user_attr("mlp_arch_lookup", [list(a) for a in architectures])
    return {
        "arch_idx": trial.suggest_int("arch_idx", 0, len(architectures) - 1),
        "_arch_lookup": architectures,
        "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [1024, 2048, 4096]),
    }


def lgb_search_space(trial) -> dict:
    """LightGBM space mirrored to hgbm_search_space — 6 params, matched ranges."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_categorical("max_depth", [4, 6, 8, 10, -1]),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 50, 500, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 1.0, log=True),
    }


### Script 06 — Tiered Search Spaces And Registry

Narrow Tier-1 spaces plus the registry used by `resolve_space`.


In [43]:
def rf_tier1_space(trial) -> dict:
    return {
        "n_estimators": 400,
        "max_depth": trial.suggest_categorical("max_depth", [6, 8, 10, 12, 15]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 100, 1000, log=True),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3]),
    }


def hgbm_tier1_space(trial) -> dict:
    return {
        "max_iter": trial.suggest_int("max_iter", 50, 250, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.3, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 127, log=True),
        "max_depth": "none",
        "min_samples_leaf": 100,
        "l2_regularization": 0.0,
    }


def lgb_tier1_space(trial) -> dict:
    return {
        "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": 800,
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 500, log=True),
        "feature_fraction": 1.0,
        "bagging_fraction": 1.0,
        "bagging_freq": 0,
        "lambda_l1": 0.0,
        "lambda_l2": 0.0,
    }


def mlp_tier1_space(trial) -> dict:
    architectures = [(64,), (128,), (64, 32), (128, 64), (256,), (256, 128)]
    trial.set_user_attr("mlp_arch_lookup", [list(a) for a in architectures])
    return {
        "arch_idx": trial.suggest_int("arch_idx", 0, len(architectures) - 1),
        "_arch_lookup": architectures,
        "activation": "relu",
        "alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
        "learning_rate_init": 1e-3,
        "batch_size": 2048,
    }


# Named-space registry. Maps `--space-name` to the function that defines
# the search. Backward-compat: bare model name resolves to the canonical
# full space so old invocations like `--models random_forest` still work.
SEARCH_SPACES: dict = {
    "random_forest": rf_search_space,
    "random_forest_tier1": rf_tier1_space,
    "random_forest_tier2": rf_search_space,  # full space = tier1+tier2
    "hist_gbm": hgbm_search_space,
    "hist_gbm_tier1": hgbm_tier1_space,
    "hist_gbm_tier2": hgbm_search_space,
    "lightgbm": lgb_search_space,
    "lightgbm_tier1": lgb_tier1_space,
    "lightgbm_tier2": lgb_search_space,
    "mlp_sklearn": mlp_search_space,
    "mlp_sklearn_tier1": mlp_tier1_space,
    "mlp_sklearn_tier2": mlp_search_space,
}


def resolve_space(model_name: str, space_name: str | None):
    """Return the search-space function for (model, optional space-name).

    If `space_name` is None, falls back to the canonical full space named
    after the model (backward compatibility with the old CLI).
    """
    name = space_name if space_name else model_name
    if name not in SEARCH_SPACES:
        raise SystemExit(
            f"unknown --space-name {name!r}. Known: {sorted(SEARCH_SPACES)}"
        )
    return SEARCH_SPACES[name], name


def json_safe_params(params: dict) -> dict:
    """Return params with helper-only objects removed and tuples made JSON-safe."""
    safe = {}
    for key, value in params.items():
        if key == "_arch_lookup":
            continue
        if isinstance(value, tuple):
            safe[key] = list(value)
        else:
            safe[key] = value
    return safe


### Script 06 — Tuned Estimator Factories

Builds estimators from sampled/best parameters.


In [44]:
def make_rf(p: dict):
    return RandomForestClassifier(
        n_estimators=p["n_estimators"], max_depth=p["max_depth"],
        min_samples_leaf=p["min_samples_leaf"], max_features=p["max_features"],
        n_jobs=N_JOBS, class_weight="balanced", random_state=RANDOM_SEED)


def make_hgbm(p: dict):
    md = None if p["max_depth"] == "none" else p["max_depth"]
    # n_jobs is not exposed by HistGradientBoostingClassifier; parallelism is
    # controlled by OMP_NUM_THREADS, which the threading-setup cell pins to N_JOBS.
    return HistGradientBoostingClassifier(
        max_iter=p["max_iter"], learning_rate=p["learning_rate"],
        max_depth=md, max_leaf_nodes=p["max_leaf_nodes"],
        min_samples_leaf=p["min_samples_leaf"], l2_regularization=p["l2_regularization"],
        class_weight="balanced", random_state=RANDOM_SEED)


def make_mlp(p: dict):
    # MLPClassifier has no n_jobs; matmul-bound. Speed comes from OPENBLAS_NUM_THREADS,
    # also pinned to N_JOBS in the threading-setup cell.
    arch = p["_arch_lookup"][p["arch_idx"]]
    return MLPClassifier(
        hidden_layer_sizes=arch, activation=p["activation"], solver="adam",
        alpha=p["alpha"], batch_size=p["batch_size"],
        learning_rate_init=p["learning_rate_init"], max_iter=50,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=5,
        random_state=RANDOM_SEED)


def make_lgb(p: dict):
    if not LIGHTGBM_AVAILABLE:
        raise SystemExit(
            "lightgbm is not installed in py312. Run "
            "`conda activate py312 && pip install lightgbm` and re-run, or "
            "drop lightgbm from the --models list."
        )
    return lgb.LGBMClassifier(
        num_leaves=p["num_leaves"],
        learning_rate=p["learning_rate"],
        n_estimators=p["n_estimators"],
        min_data_in_leaf=p["min_data_in_leaf"],
        feature_fraction=p.get("feature_fraction", 1.0),
        bagging_fraction=p.get("bagging_fraction", 1.0),
        bagging_freq=p.get("bagging_freq", 0),
        lambda_l1=p.get("lambda_l1", 0.0),
        lambda_l2=p.get("lambda_l2", 0.0),
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        num_threads=N_JOBS,  # lightgbm honours this directly; redundant with n_jobs but explicit
        verbose=-1,
    )


### Script 06 — Data Loading And Objectives

Loads train/test matrices and defines grouped K-fold or grouped holdout Optuna objectives.


In [45]:
def load_xy() -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.DataFrame, pd.Series, list[str]]:
    """Load consolidated parquet, split, get the canonical feature list."""
    df = pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet")
    feature_cols = json.loads((OUTPUTS_DIR / "data" / "feature_cols.json").read_text())
    train = df[df["split"] == "train"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)
    X_train = train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_train = train[TARGET].astype(int)
    y_test = test[TARGET].astype(int)
    g_train = train["market_id"]
    return X_train, y_train, g_train, X_test, y_test, feature_cols


# ----------------------------------------------------------------------------
# Objective + tuning loop per model
# ----------------------------------------------------------------------------

def objective_kfold(trial, make_fn, params: dict, X, y, groups, n_folds: int, scale: bool) -> float:
    """Mean OOF AUC across folds, the value Optuna maximises.

    Reports the running fold-mean to the trial after each fold so MedianPruner
    can compare partial progress against other trials at the same step. Stashes
    the per-fold AUC list and fold std on the trial as user attrs so the loop
    log can record both in `tuning_log.jsonl`.
    """
    # what: same GroupKFold protocol used by 03_train_models so OOF AUC is directly comparable
    gkf = GroupKFold(n_splits=n_folds)
    aucs: list[float] = []
    for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        if scale:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X.iloc[tr_idx])
            X_va = scaler.transform(X.iloc[va_idx])
        else:
            X_tr, X_va = X.iloc[tr_idx].values, X.iloc[va_idx].values
        clf = make_fn(params).fit(X_tr, y.iloc[tr_idx])
        if hasattr(clf, "predict_proba"):
            preds = clf.predict_proba(X_va)[:, 1]
        else:
            d = clf.decision_function(X_va)
            preds = (d - d.min()) / (d.max() - d.min() + 1e-9)
        aucs.append(float(roc_auc_score(y.iloc[va_idx], preds)))
        # Real intermediate reporting: MedianPruner compares the running
        # fold-mean against other trials' running mean at the same step
        # and prunes weak trials before the next fold runs.
        running_mean = float(np.mean(aucs))
        trial.report(running_mean, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr("fold_aucs", aucs)
            trial.set_user_attr("fold_std", float(np.std(aucs)) if len(aucs) > 1 else float("nan"))
            trial.set_user_attr("eval_mode", "kfold")
            raise optuna.TrialPruned()
    trial.set_user_attr("fold_aucs", aucs)
    trial.set_user_attr("fold_std", float(np.std(aucs)))
    trial.set_user_attr("eval_mode", "kfold")
    return float(np.mean(aucs))


def objective_holdout(trial, make_fn, params: dict, X, y, groups, scale: bool) -> float:
    """Single GroupShuffleSplit holdout AUC, used for slow models like MLP.

    NOTE: MLP best_oof_auc reported here is NOT directly comparable to RF/HGBM's
    5-fold GroupKFold mean AUC (which is what 03_train_models reports). Treat as
    a coarse holdout estimate, and surface the caveat in the methodology section
    of the report when citing tuning results across model families.

    `fold_std` is set to NaN on the trial since holdout has only one score; the
    log records this so downstream readers do not mistake holdout for KFold.
    """
    # what: 80/20 group-aware holdout; 5x faster than full KFold; trade rigour for runtime
    # why: MLP fit takes ~5 min per fold -> 25 min per trial -> 30 trials = 12.5 hr (too slow)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
    tr_idx, va_idx = next(splitter.split(X, y, groups))
    if scale:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X.iloc[tr_idx])
        X_va = scaler.transform(X.iloc[va_idx])
    else:
        X_tr, X_va = X.iloc[tr_idx].values, X.iloc[va_idx].values
    clf = make_fn(params).fit(X_tr, y.iloc[tr_idx])
    preds = clf.predict_proba(X_va)[:, 1] if hasattr(clf, "predict_proba") \
        else clf.decision_function(X_va)
    auc = float(roc_auc_score(y.iloc[va_idx], preds))
    trial.set_user_attr("fold_aucs", [auc])
    trial.set_user_attr("fold_std", float("nan"))
    trial.set_user_attr("eval_mode", "holdout")
    return auc


def attach_mlp_arch_lookup(params: dict, arch_lookup) -> dict:
    """Attach the exact architecture lookup used by an MLP search space."""
    if arch_lookup is None:
        raise SystemExit(
            "MLP refit aborted: the search space did not stash "
            "`mlp_arch_lookup` on the trial. Update the space function "
            "to call `trial.set_user_attr('mlp_arch_lookup', architectures)`."
        )
    out = dict(params)
    out["_arch_lookup"] = [tuple(a) for a in arch_lookup]
    return out


### Script 06 — Refit And Tuning Loop

Writes tuned OOF/test predictions, best parameters, and study history to `outputs/tuning/<model>/`.


In [46]:
def write_refit_outputs(model_name: str, make_fn, best: dict, scale: bool,
                         X_train, y_train, g_train, X_test, y_test,
                         out_dir: Path, *, best_oof_auc: float | None,
                         suggested_params: dict | None,
                         n_trials: int | None) -> None:
    """Refit best params, write tuned test + tuned OOF predictions."""
    print("  refitting with best params on full train ...")
    if scale:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
    else:
        X_tr, X_te = X_train.values, X_test.values
    clf = make_fn(best).fit(X_tr, y_train)
    raw_tuned = clf.predict_proba(X_te)[:, 1]
    np.savez_compressed(out_dir / "preds_test_tuned.npz", raw=raw_tuned.astype("float32"))

    # what: also generate tuned OOF predictions so the bridge in
    # `promote_tuned_preds.py` can wire 04_calibration.py and
    # 05_backtest.py to the tuned lineup. Calibrating tuned test preds
    # requires tuned OOF; baseline OOF would mismap the score distribution.
    print("  generating tuned OOF preds for downstream calibration ...")
    oof_tuned = np.zeros(len(y_train), dtype="float32")
    gkf_oof = GroupKFold(n_splits=N_FOLDS)
    for tr_idx, va_idx in gkf_oof.split(X_train, y_train, g_train):
        if scale:
            sc = StandardScaler()
            Xtr_oof = sc.fit_transform(X_train.iloc[tr_idx])
            Xva_oof = sc.transform(X_train.iloc[va_idx])
        else:
            Xtr_oof = X_train.iloc[tr_idx].values
            Xva_oof = X_train.iloc[va_idx].values
        fold_clf = make_fn(best).fit(Xtr_oof, y_train.iloc[tr_idx])
        if hasattr(fold_clf, "predict_proba"):
            p = fold_clf.predict_proba(Xva_oof)[:, 1]
        else:
            d = fold_clf.decision_function(Xva_oof)
            p = (d - d.min()) / (d.max() - d.min() + 1e-9)
        oof_tuned[va_idx] = p.astype("float32")
    np.save(out_dir / "preds_oof_tuned.npy", oof_tuned)

    tuned_auc = float(roc_auc_score(y_test, raw_tuned))

    # what: compare against the default model's raw test AUC (already saved by 03_train_models)
    default_path = OUTPUTS_DIR / "models" / model_name / "preds_test.npz"
    delta = None
    default_auc = None
    if default_path.exists():
        default_raw = np.load(default_path)["raw"]
        default_auc = float(roc_auc_score(y_test, default_raw))
        delta = tuned_auc - default_auc
        print(f"  test AUC: tuned={tuned_auc:.4f}  default={default_auc:.4f}  delta={delta:+.4f}")
    (out_dir / "comparison_vs_default.json").write_text(json.dumps({
        "model": model_name,
        "best_oof_auc": best_oof_auc,
        "test_auc_tuned": tuned_auc,
        "test_auc_default": default_auc,
        "delta_test_auc": delta,
        "best_params": json_safe_params(best),
        "suggested_params": suggested_params,
        "n_trials": n_trials,
    }, indent=2))


def tune_one_model(model_name: str, n_trials: int,
                   space_name: str | None = None,
                   time_budget_min: float | None = None,
                   no_test_eval: bool = False,
                   fixed_params_file: Path | None = None) -> None:
    """Optuna TPE loop for one model.

    By default refits the best params on full train and writes test predictions
    + test AUC, preserving the original one-shot behaviour. The autonomous
    tuning loop passes `no_test_eval=True` to keep test data out of the loop
    per the anti-leakage protocol in `tuning/tuning_plan.md`.

    `time_budget_min` is a soft cap: Optuna only checks it between trials, so
    a single long-running trial can overshoot before the loop exits.

    `fixed_params_file` skips the Optuna search and replays a saved
    best_params.json payload. Use this only in the final evaluation pass.
    """
    print("\n" + "=" * 60)
    print(f"Tuning {model_name}"
          + (f" (space={space_name})" if space_name else "")
          + (f" trials<={n_trials}, timeout<={time_budget_min}min"
             if time_budget_min else f" ({n_trials} trials)"))
    print("=" * 60)

    # what: per-model wiring (factory + scaling + holdout-vs-kfold default)
    if model_name == "random_forest":
        make_fn, scale, holdout = make_rf, False, False
    elif model_name == "hist_gbm":
        make_fn, scale, holdout = make_hgbm, False, False
    elif model_name == "mlp_sklearn":
        make_fn, scale, holdout = make_mlp, True, True
    elif model_name == "lightgbm":
        if not LIGHTGBM_AVAILABLE:
            print("  [lightgbm] not installed in py312, deferring per the loop's "
                  "fall-through rule. Install with `pip install lightgbm`.")
            return
        make_fn, scale, holdout = make_lgb, False, False
    else:
        raise SystemExit(f"unknown model: {model_name}")

    space, resolved_space_name = resolve_space(model_name, space_name)

    X_train, y_train, g_train, X_test, y_test, _ = load_xy()
    out_dir = OUTPUTS_DIR / "tuning" / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    if fixed_params_file is not None:
        payload = json.loads(fixed_params_file.read_text())
        best = dict(payload["best_params"])
        if model_name == "mlp_sklearn":
            best = attach_mlp_arch_lookup(best, payload.get("mlp_arch_lookup"))
        print(f"  replaying fixed params from {fixed_params_file}")
        write_refit_outputs(
            model_name, make_fn, best, scale,
            X_train, y_train, g_train, X_test, y_test, out_dir,
            best_oof_auc=payload.get("best_oof_auc"),
            suggested_params=payload.get("suggested_params"),
            n_trials=payload.get("n_trials_completed"),
        )
        return

    # what: define the optuna objective in closure form so the search space + factory are bound
    def objective(trial):
        params = space(trial)
        # Optuna's `best_params` contains only suggested parameters, not fixed
        # constants returned by named Tier-1 spaces. Store the full dict so
        # final refit and logging can faithfully replay the exact study winner.
        trial.set_user_attr("full_params", json_safe_params(params))
        if holdout:
            return objective_holdout(trial, make_fn, params, X_train, y_train, g_train, scale=scale)
        return objective_kfold(trial, make_fn, params, X_train, y_train, g_train,
                                n_folds=N_FOLDS, scale=scale)

    # what: TPE sampler with median pruner. Pruning is now real because the
    # objective calls trial.report/should_prune between folds.
    sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner,
                                 study_name=f"{model_name}::{resolved_space_name}")
    t0 = time.time()
    timeout_sec = float(time_budget_min) * 60.0 if time_budget_min else None
    study.optimize(objective, n_trials=n_trials, timeout=timeout_sec, show_progress_bar=False)
    wall_clock_min = (time.time() - t0) / 60.0
    print(f"  done in {wall_clock_min:.1f} min  best AUC = {study.best_value:.4f}  "
          f"trials_completed = {len(study.trials)}")

    # what: persist study history + best params (always, regardless of test-eval mode)
    history_rows = [{"trial": t.number, "value": t.value, "state": str(t.state),
                     "duration_sec": t.duration.total_seconds() if t.duration else None,
                     **t.params} for t in study.trials]
    pd.DataFrame(history_rows).to_csv(out_dir / "study_history.csv", index=False)

    best_trial = study.best_trial
    fold_aucs = best_trial.user_attrs.get("fold_aucs")
    fold_std = best_trial.user_attrs.get("fold_std")
    eval_mode = best_trial.user_attrs.get("eval_mode", "kfold")
    full_best_params = best_trial.user_attrs.get("full_params")
    if full_best_params is None:
        full_best_params = dict(study.best_params)
    mlp_arch_lookup = best_trial.user_attrs.get("mlp_arch_lookup")

    best_params_payload = {
        "model": model_name,
        "space_name": resolved_space_name,
        "best_oof_auc": float(study.best_value),
        "best_oof_auc_fold_std": fold_std,
        "best_oof_fold_aucs": fold_aucs,
        "eval_mode": eval_mode,
        "best_params": full_best_params,
        "suggested_params": study.best_params,
        "mlp_arch_lookup": mlp_arch_lookup,
        "n_trials_completed": len(study.trials),
        "wall_clock_min": wall_clock_min,
        "n_trials_requested": n_trials,
        "time_budget_min": time_budget_min,
    }
    (out_dir / "best_params.json").write_text(json.dumps(best_params_payload, indent=2))

    if no_test_eval:
        # CV-only mode for the autonomous loop. Refit and test-AUC are
        # forbidden inside the loop per tuning_plan.md anti-leakage rule.
        print("  --no-test-eval set, skipping refit/test-AUC block. "
              "Run the post-loop final-evaluation pass for honest test metrics.")
        return

    # what: refit best params on FULL train, predict on test (one-shot mode)
    best = dict(full_best_params)
    if model_name == "mlp_sklearn":
        # Recover the exact architecture list the study searched against,
        # not the canonical full-space list. Tier-1 and full spaces disagree
        # on the high-index architectures, so falling back to a hardcoded
        # list would refit the wrong network for arch_idx >= 4.
        best = attach_mlp_arch_lookup(best, mlp_arch_lookup)
    write_refit_outputs(
        model_name, make_fn, best, scale,
        X_train, y_train, g_train, X_test, y_test, out_dir,
        best_oof_auc=float(study.best_value),
        suggested_params=study.best_params,
        n_trials=n_trials,
    )


### Script 06 — Stage Entrypoint

Default tuning plan for Random Forest and HistGBM, preserved behind a gate.


In [47]:
def main_06() -> int:
    # NOTEBOOK: argparse replaced with SimpleNamespace using argparse defaults.
    # Aligned with Pontus's tuning plan: 3 models, 52 trials each.
    args = SimpleNamespace(
        models=["random_forest", "hist_gbm", "mlp_sklearn"],
        n_trials=52,
        space_name=None,
        time_budget_min=None,
        no_test_eval=False,
        fixed_params_file=None,
    )

    print("=" * 60)
    print(f"Stage 6, Optuna tuning ({len(args.models)} model(s), "
          f"trials<={args.n_trials}"
          + (f", timeout<={args.time_budget_min}min" if args.time_budget_min else "")
          + (", no-test-eval" if args.no_test_eval else "")
          + (f", fixed={args.fixed_params_file}" if args.fixed_params_file else "")
          + ")")
    print("=" * 60)
    for m in args.models:
        try:
            tune_one_model(
                m,
                n_trials=args.n_trials,
                space_name=args.space_name,
                time_budget_min=args.time_budget_min,
                no_test_eval=args.no_test_eval,
                fixed_params_file=args.fixed_params_file,
            )
        except Exception as e:
            print(f"  [{m}] FAILED: {e}")

    if args.no_test_eval:
        print("\nStage 6 complete (CV-only). best_params.json and study_history.csv "
              "in outputs/tuning/<model>/. Run the post-loop final-evaluation pass for test metrics.")
    else:
        print("\nStage 6 complete. Tuned predictions are in outputs/tuning/<model>/preds_test_tuned.npz.")
        print("Re-run 04_calibration.py and 05_backtest.py to put tuned predictions in the headline.")
    return 0


### Script 06 — Run Default Tuning (Gated)

Default is no-op. Set `RUN_TUNING = True` to tune the default model list in `main_06()`.


In [48]:
np.random.seed(RANDOM_SEED)
# Gated: optional and slow. Flip to True to run the default tuning plan.
RUN_TUNING = True
if RUN_TUNING and OPTUNA_AVAILABLE:
    main_06()
elif RUN_TUNING and not OPTUNA_AVAILABLE:
    print("Optuna is not available; install optuna or run the setup dependency cell first.")


Stage 6, Optuna tuning (3 model(s), trials<=52)

Tuning random_forest (52 trials)


[W 2026-05-15 14:12:18,385] Trial 17 failed with parameters: {'n_estimators': 350, 'max_depth': 15, 'min_samples_leaf': 102, 'max_features': 0.3} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/envs/ml/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/7w/_356msz143sf3hv722cv7wzh0000gn/T/ipykernel_59436/3831734588.py", line 136, in objective
    return objective_kfold(trial, make_fn, params, X_train, y_train, g_train,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/7w/_356msz143sf3hv722cv7wzh0000gn/T/ipykernel_59436/3569741312.py", line 37, in objective_kfold
    clf = make_fn(params).fit(X_tr, y.iloc[tr_idx])
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py", line 1336, in wrapp

KeyboardInterrupt: 

### Script 06 — Tune LightGBM Only (Gated)

This replaces the old final all-in-one LightGBM cell's tuning step. Runtime is roughly 50-90 minutes for 52 trials.


In [ ]:
np.random.seed(RANDOM_SEED)
# Gated: tune only LightGBM using the canonical full search space.
RUN_LIGHTGBM_TUNING = True
if RUN_LIGHTGBM_TUNING and OPTUNA_AVAILABLE and LIGHTGBM_AVAILABLE:
    print("=" * 60)
    print("Tuning LightGBM (52 trials, 5-fold GroupKFold + MedianPruner)")
    print("=" * 60)
    tune_one_model(
        "lightgbm",
        n_trials=52,
        space_name=None,
        time_budget_min=None,
        no_test_eval=False,
        fixed_params_file=None,
    )
elif RUN_LIGHTGBM_TUNING and not OPTUNA_AVAILABLE:
    print("Optuna is not available; install optuna or run the setup dependency cell first.")
elif RUN_LIGHTGBM_TUNING and not LIGHTGBM_AVAILABLE:
    print("LightGBM is not available; install lightgbm or run the setup dependency cell first.")


## Appendix — Promote Tuned Predictions And Refresh Outputs

Manual bridge for copying tuned predictions into canonical model folders, then refreshing calibration and backtest artifacts. All cells are gated off by default.


### Appendix — Promotion Constants

Defines source/target directories, prediction file pairs, and derived files that must be cleared after promotion or restore.


In [ ]:
import argparse
import shutil
import sys
from pathlib import Path

SUBMISSION_DIR = NOTEBOOK_DIR
TUNING_DIR = SUBMISSION_DIR / "outputs" / "tuning"
MODELS_DIR = SUBMISSION_DIR / "outputs" / "models"

PAIRS = [
    ("preds_oof_tuned.npy", "preds_oof.npy", "preds_oof_baseline.npy"),
    ("preds_test_tuned.npz", "preds_test.npz", "preds_test_baseline.npz"),
]

DERIVED_FILES = [
    "preds_test_cal.npz",
    "isotonic.joblib",
]


### Appendix — Promotion Helpers

Discovers tuned models, promotes tuned predictions, and restores baseline predictions when backups exist.


In [ ]:
def clear_derived(model_dir: Path) -> None:
    """Remove calibration outputs that no longer match the raw/OOF predictions."""
    for name in DERIVED_FILES:
        path = model_dir / name
        if path.exists():
            path.unlink()


def discover_models(explicit: list[str] | None) -> list[str]:
    if explicit:
        return explicit
    if not TUNING_DIR.exists():
        return []
    return sorted(p.name for p in TUNING_DIR.iterdir() if p.is_dir())


def promote(models: list[str]) -> int:
    promoted = 0
    skipped: list[str] = []
    for m in models:
        src = TUNING_DIR / m
        dst = MODELS_DIR / m
        if not src.is_dir():
            skipped.append(f"{m}: no outputs/tuning/{m}/")
            continue
        if not dst.is_dir():
            skipped.append(f"{m}: no outputs/models/{m}/ (run 03_train_models first)")
            continue
        missing = [tuned for tuned, _, _ in PAIRS if not (src / tuned).exists()]
        if missing:
            skipped.append(f"{m}: missing {missing} under outputs/tuning/{m}/")
            continue
        for tuned, canonical, backup in PAIRS:
            tuned_path = src / tuned
            canonical_path = dst / canonical
            backup_path = dst / backup
            if canonical_path.exists() and not backup_path.exists():
                shutil.copy2(canonical_path, backup_path)
            shutil.copy2(tuned_path, canonical_path)
        clear_derived(dst)
        promoted += 1
        print(f"  promoted {m}: tuned preds now live at outputs/models/{m}/ "
              "and stale calibration files were cleared")
    if skipped:
        print("\nSkipped:")
        for line in skipped:
            print(f"  {line}")
    print(f"\nPromoted {promoted} model(s). Re-run 04_calibration.py and 05_backtest.py "
          f"to refresh calibrated metrics and the ROI heatmap on the tuned lineup.")
    return 0 if promoted > 0 else 1


def restore(models: list[str]) -> int:
    restored = 0
    skipped: list[str] = []
    for m in models:
        dst = MODELS_DIR / m
        if not dst.is_dir():
            skipped.append(f"{m}: no outputs/models/{m}/")
            continue
        any_done = False
        for _, canonical, backup in PAIRS:
            backup_path = dst / backup
            canonical_path = dst / canonical
            if backup_path.exists():
                shutil.copy2(backup_path, canonical_path)
                any_done = True
        if any_done:
            clear_derived(dst)
            restored += 1
            print(f"  restored {m}: outputs/models/{m}/ now back to baseline preds "
                  "and stale calibration files were cleared")
        else:
            skipped.append(f"{m}: no *_baseline.* backups found, nothing to restore")
    if skipped:
        print("\nSkipped:")
        for line in skipped:
            print(f"  {line}")
    print(f"\nRestored {restored} model(s) to baseline.")
    return 0


### Appendix — Baseline Backtest Snapshot

Copies the current baseline `outputs/backtest/sensitivity.csv` before promotion, so the paper-figure section can compute ROI deltas after tuned predictions are promoted and Scripts 04-05 are refreshed.


In [ ]:

def snapshot_baseline_backtest() -> Path | None:
    """Snapshot baseline backtest sensitivity before tuned predictions overwrite canonical outputs."""
    src = OUTPUTS_DIR / "backtest" / "sensitivity.csv"
    if not src.exists():
        print("  baseline snapshot skipped: outputs/backtest/sensitivity.csv does not exist yet")
        return None

    dst_dir = OUTPUTS_DIR / "_baseline_snapshot" / "backtest"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "sensitivity.csv"
    if dst.exists():
        print(f"  baseline snapshot already exists at {dst}")
        return dst

    shutil.copy2(src, dst)
    print(f"  snapshotted baseline sensitivity -> {dst}")
    return dst


### Appendix — Promotion Entrypoint

Notebook-friendly equivalent of the original promotion script entrypoint.


In [ ]:
def main_promote() -> int:
    # NOTEBOOK: argparse replaced with SimpleNamespace.
    args = SimpleNamespace(models=None, restore=False)

    models = discover_models(args.models)
    if not models:
        print("No models found in outputs/tuning/. Run 06_tuning_optuna.py first.")
        return 1

    if args.restore:
        return restore(models)
    snapshot_baseline_backtest()
    return promote(models)


### Appendix — Promote Or Restore Predictions (Gated)

Default is no-op. Set `RUN_PROMOTE = True` after tuning. By default this targets LightGBM; set `PROMOTE_MODELS = None` to discover all tuned models.


In [ ]:
# Gated: promote tuned predictions into outputs/models/<model>/, or restore baseline backups.
# PROMOTE_MODELS = None auto-discovers every folder under outputs/tuning/, so all four
# tuned models (RF, hist_gbm, MLP, LightGBM) are promoted in one pass.
RUN_PROMOTE = True
PROMOTE_MODELS = None
RESTORE_BASELINE = False

if RUN_PROMOTE:
    models = discover_models(PROMOTE_MODELS)
    if RESTORE_BASELINE:
        restore(models)
    else:
        snapshot_baseline_backtest()
        promote(models)


### Appendix — Refresh Calibration After Promotion (Gated)

Default is no-op. Run this after promotion so `outputs/metrics/` and calibrated prediction files match the promoted model predictions.


In [ ]:
np.random.seed(RANDOM_SEED)
RUN_REFRESH_CALIBRATION = True
if RUN_REFRESH_CALIBRATION:
    print("=" * 60)
    print("Re-running Script 04 (calibration) on current model predictions")
    print("=" * 60)
    main_04()


### Appendix — Refresh Backtest After Promotion (Gated)

Default is no-op. Run this after calibration refresh to update `falsification.json`, `overview.png`, and backtest diagnostics.


In [ ]:
np.random.seed(RANDOM_SEED)
RUN_REFRESH_BACKTEST = True
if RUN_REFRESH_BACKTEST:
    print("=" * 60)
    print("Re-running Script 05 (backtest) on current calibrated predictions")
    print("=" * 60)
    main_05()

    cal_path = OUTPUTS_DIR / "metrics" / "calibration_summary.csv"
    if cal_path.exists():
        cal = pd.read_csv(cal_path)
        lgb_rows = cal[cal["model"] == "lightgbm"]
        if not lgb_rows.empty:
            lgb_row = lgb_rows.iloc[0]
            print()
            print("=" * 60)
            print("LightGBM summary")
            print("=" * 60)


## Generate Paper Figures

Generates the visual set referenced by the paper and writes paper-facing PNGs to `outputs/figures/`. Existing Script 04/05 figures are copied into that directory, new EDA/model/tuning figures are rendered there, and missing optional tuning artifacts are skipped with explicit messages.


### Generate Paper Figures — Helpers

Definitions only: shared style, save/copy/display helpers, parquet column readers, AUC/prediction utilities, and one generator per paper figure.


In [ ]:

from IPython.display import Image as IPyImage, display
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score as _roc_auc_score
import pyarrow.parquet as pq
import textwrap

TARGET = globals().get("TARGET", "bet_correct")
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DPI = 300
HEADLINE_TUNED_MODELS = ["random_forest", "hist_gbm", "mlp_sklearn", "lightgbm"]

_MODELING_SCHEMA_COLUMNS: set[str] | None = None
_BACKTEST_SCHEMA_COLUMNS: set[str] | None = None


def figure_style() -> None:
    """Apply the shared visual theme before rendering paper figures."""
    try:
        apply_theme()
    except NameError:
        sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({"savefig.dpi": FIGURE_DPI, "figure.dpi": 120})


def figure_path(filename: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    return FIGURES_DIR / filename


def display_figure(path: Path) -> None:
    if path.exists():
        display(IPyImage(filename=str(path)))


def save_fig(fig, filename: str) -> Path:
    path = figure_path(filename)
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {path}")
    display_figure(path)
    return path


def copy_paper_figure(src: Path, filename: str) -> Path | None:
    dst = figure_path(filename)
    if not src.exists():
        print(f"  skipped {filename}: missing source {src}")
        return None
    shutil.copy2(src, dst)
    print(f"  copied {src.name} -> {dst}")
    display_figure(dst)
    return dst


def skip_figure(filename: str, reason: str) -> None:
    print(f"  skipped {filename}: {reason}")


def _modeling_columns() -> set[str]:
    global _MODELING_SCHEMA_COLUMNS
    if _MODELING_SCHEMA_COLUMNS is None:
        _MODELING_SCHEMA_COLUMNS = set(
            pq.ParquetFile(DATA_DIR / "consolidated_modeling_data.parquet").schema.names
        )
    return _MODELING_SCHEMA_COLUMNS


def _backtest_columns() -> set[str]:
    global _BACKTEST_SCHEMA_COLUMNS
    if _BACKTEST_SCHEMA_COLUMNS is None:
        _BACKTEST_SCHEMA_COLUMNS = set(
            pq.ParquetFile(DATA_DIR / "backtest_context.parquet").schema.names
        )
    return _BACKTEST_SCHEMA_COLUMNS


def read_modeling_columns(columns: list[str]) -> pd.DataFrame:
    available = _modeling_columns()
    requested = list(dict.fromkeys(columns))
    present = [c for c in requested if c in available]
    missing = [c for c in requested if c not in available]
    if missing:
        print(f"  note: missing modeling columns skipped: {missing}")
    if not present:
        raise ValueError("No requested modeling columns are available")
    return pd.read_parquet(DATA_DIR / "consolidated_modeling_data.parquet", columns=present)


def read_backtest_columns(columns: list[str]) -> pd.DataFrame:
    available = _backtest_columns()
    requested = list(dict.fromkeys(columns))
    present = [c for c in requested if c in available]
    missing = [c for c in requested if c not in available]
    if missing:
        print(f"  note: missing backtest columns skipped: {missing}")
    if not present:
        raise ValueError("No requested backtest columns are available")
    return pd.read_parquet(DATA_DIR / "backtest_context.parquet", columns=present)


def load_feature_cols() -> list[str]:
    path = OUTPUTS_DIR / "data" / "feature_cols.json"
    if not path.exists():
        raise FileNotFoundError("Run Script 01 first: outputs/data/feature_cols.json is missing")
    return json.loads(path.read_text())


def clean_label(label: str, width: int = 24) -> str:
    text = str(label).replace("_", " ")
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False))


def short_market_label(label: str, width: int = 18) -> str:
    text = str(label)
    if len(text) <= width:
        return text
    return text[: width - 1] + "…"


def safe_auc_score(y_true, score) -> float:
    y = np.asarray(y_true)
    x = pd.to_numeric(pd.Series(score), errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(x) & pd.notna(y)
    y = y[mask].astype(int)
    x = x[mask]
    if len(y) < 20 or len(np.unique(y)) < 2 or np.nanstd(x) == 0:
        return np.nan
    return float(_roc_auc_score(y, x))


def load_pred_array(path: Path) -> np.ndarray:
    obj = np.load(path)
    if isinstance(obj, np.lib.npyio.NpzFile):
        try:
            for key in ("raw", "cal", "arr_0", "pred", "preds"):
                if key in obj.files:
                    return np.asarray(obj[key], dtype=float)
            return np.asarray(obj[obj.files[0]], dtype=float)
        finally:
            obj.close()
    return np.asarray(obj, dtype=float)


def generate_figure_02_baseline_vs_tuned() -> Path | None:
    filename = "figure_02_baseline_vs_tuned.png"
    y = read_modeling_columns(["split", TARGET])
    y_test = y.loc[y["split"] == "test", TARGET].astype(int).reset_index(drop=True).values

    rows = []
    for model in HEADLINE_TUNED_MODELS:
        model_dir = OUTPUTS_DIR / "models" / model
        tuning_dir = OUTPUTS_DIR / "tuning" / model
        promoted_baseline = model_dir / "preds_test_baseline.npz"
        promoted_tuned = model_dir / "preds_test.npz"
        unpromoted_tuned = tuning_dir / "preds_test_tuned.npz"

        if promoted_baseline.exists() and promoted_tuned.exists():
            baseline_path = promoted_baseline
            tuned_path = promoted_tuned
            source = "promoted"
        elif promoted_tuned.exists() and unpromoted_tuned.exists():
            baseline_path = promoted_tuned
            tuned_path = unpromoted_tuned
            source = "unpromoted tuning output"
        else:
            print(f"  figure 02: skipped {model}; need baseline+tuned test predictions")
            continue

        baseline = load_pred_array(baseline_path)
        tuned = load_pred_array(tuned_path)
        if len(baseline) != len(y_test) or len(tuned) != len(y_test):
            print(f"  figure 02: skipped {model}; prediction length mismatch")
            continue
        rows.append({
            "model": model,
            "baseline_auc": safe_auc_score(y_test, baseline),
            "tuned_auc": safe_auc_score(y_test, tuned),
            "source": source,
        })

    if not rows:
        skip_figure(filename, "no models have both baseline and tuned test predictions")
        return None

    df = pd.DataFrame(rows).dropna(subset=["baseline_auc", "tuned_auc"])
    if df.empty:
        skip_figure(filename, "all baseline/tuned AUC comparisons were invalid")
        return None

    figure_style()
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    x = np.arange(len(df))
    width = 0.36
    colors = [PAL_10[3], PAL_10[6]] if "PAL_10" in globals() else sns.color_palette("rocket", 7)[3:7:3]
    ax.bar(x - width / 2, df["baseline_auc"], width, label="Baseline", color=colors[0])
    ax.bar(x + width / 2, df["tuned_auc"], width, label="Tuned", color=colors[1])

    y_min = max(0.45, float(df[["baseline_auc", "tuned_auc"]].min().min()) - 0.015)
    y_max = min(1.0, float(df[["baseline_auc", "tuned_auc"]].max().max()) + 0.03)
    ax.set_ylim(y_min, y_max)
    ax.set_ylabel("Test ROC-AUC")
    ax.set_xticks(x)
    ax.set_xticklabels([clean_label(m, 16) for m in df["model"]])
    ax.legend(frameon=False, ncols=2, loc="upper left")
    ax.set_title("Baseline vs tuned test AUC")
    for i, row in df.reset_index(drop=True).iterrows():
        delta = row["tuned_auc"] - row["baseline_auc"]
        y_pos = max(row["baseline_auc"], row["tuned_auc"]) + 0.004
        ax.text(i, y_pos, f"Δ {delta:+.3f}", ha="center", va="bottom", fontsize=8)
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A01_train_test_shift() -> Path | None:
    filename = "figure_A01_train_test_shift.png"
    feature_cols = [c for c in load_feature_cols() if c in _modeling_columns()]
    df = read_modeling_columns(["split", *feature_cols])
    train = df[df["split"] == "train"][feature_cols].replace([np.inf, -np.inf], np.nan)
    test = df[df["split"] == "test"][feature_cols].replace([np.inf, -np.inf], np.nan)
    z_shift = (test.mean() - train.mean()) / train.std(ddof=0).replace(0, np.nan)
    top = z_shift.reindex(z_shift.abs().sort_values(ascending=False).head(15).index).sort_values()
    if top.empty:
        skip_figure(filename, "no feature columns available")
        return None

    figure_style()
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    colors = np.where(top.values >= 0, COL_PERF_GOOD, COL_PERF_BAD) if "COL_PERF_GOOD" in globals() else None
    ax.barh([clean_label(i) for i in top.index], top.values, color=colors)
    ax.axvline(0, color="0.25", lw=0.8)
    ax.set_xlabel("Standardized test − train mean shift")
    ax.set_title("Largest train-to-test feature mean shifts")
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A02_time_to_deadline() -> Path | None:
    filename = "figure_A02_time_to_deadline.png"
    df = read_modeling_columns([TARGET, "log_time_to_deadline_hours"])
    hours = np.exp(pd.to_numeric(df["log_time_to_deadline_hours"], errors="coerce"))
    bins = [-np.inf, 1, 6, 24, 72, 168, 720, np.inf]
    labels = ["≤1h", "1-6h", "6-24h", "1-3d", "3-7d", "1-4w", ">4w"]
    df = df.assign(deadline_bucket=pd.cut(hours, bins=bins, labels=labels))
    stats = df.groupby("deadline_bucket", observed=True)[TARGET].agg(hit_rate="mean", n="size").reindex(labels)
    stats = stats.dropna(subset=["hit_rate"])
    if stats.empty:
        skip_figure(filename, "no valid time-to-deadline buckets")
        return None

    figure_style()
    fig, ax_rate = plt.subplots(figsize=(7.4, 4.2))
    ax_count = ax_rate.twinx()
    x = np.arange(len(stats))
    ax_count.bar(x, stats["n"], color="0.85", width=0.72, label="Trades")
    ax_rate.plot(x, stats["hit_rate"], color=PAL_10[5], marker="o", lw=2, label="Hit rate")
    ax_rate.axhline(0.5, color="0.25", ls="--", lw=0.8)
    ax_rate.set_xticks(x)
    ax_rate.set_xticklabels(stats.index.astype(str))
    ax_rate.set_ylabel("Bet-correct hit rate")
    ax_count.set_ylabel("Trade count")
    ax_rate.set_xlabel("Time to deadline")
    ax_rate.set_title("Hit rate and count by time-to-deadline bucket")
    clean_ax(ax_rate)
    return save_fig(fig, filename)


def generate_A03_wallet_strata() -> Path | None:
    filename = "figure_A03_wallet_strata.png"
    wallet_features = [
        "wallet_enriched", "wallet_polygon_age_at_t_days", "wallet_log_polygon_nonce_at_t",
        "wallet_log_n_inbound_at_t", "wallet_n_cex_deposits_at_t", "wallet_log_cex_usdc_cum",
        "days_from_first_usdc_to_t", "wallet_funded_by_cex_scoped",
    ]
    present = [c for c in wallet_features if c in _modeling_columns()]
    if not present:
        skip_figure(filename, "wallet feature columns are missing")
        return None
    df = read_modeling_columns([TARGET, *present])
    rows = []
    for feat in present:
        s = pd.to_numeric(df[feat], errors="coerce")
        if s.notna().sum() == 0:
            continue
        unique = s.dropna().unique()
        if len(unique) <= 2:
            groups = [("0 / no", s == 0), ("1 / yes", s == 1)]
        else:
            med = s.median()
            groups = [("low", s <= med), ("high", s > med)]
        for label, mask in groups:
            n = int(mask.sum())
            if n == 0:
                continue
            rows.append({"feature": feat, "stratum": label, "hit_rate": float(df.loc[mask, TARGET].mean()), "n": n})
    stats = pd.DataFrame(rows)
    if stats.empty:
        skip_figure(filename, "no wallet strata could be computed")
        return None

    figure_style()
    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    features = list(dict.fromkeys(stats["feature"]))
    x = np.arange(len(features))
    width = 0.36
    low = stats[stats["stratum"].isin(["low", "0 / no"])].set_index("feature").reindex(features)
    high = stats[stats["stratum"].isin(["high", "1 / yes"])].set_index("feature").reindex(features)
    ax.bar(x - width / 2, low["hit_rate"], width, label="Low / no", color=PAL_10[3])
    ax.bar(x + width / 2, high["hit_rate"], width, label="High / yes", color=PAL_10[6])
    ax.axhline(df[TARGET].mean(), color="0.25", ls="--", lw=0.8, label="Overall")
    ax.set_xticks(x)
    ax.set_xticklabels([clean_label(f, 15) for f in features], rotation=0)
    ax.set_ylabel("Bet-correct base rate")
    ax.set_title("Base rate by wallet feature stratum")
    ax.legend(frameon=False, ncols=3, loc="upper left")
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A04_per_market_side() -> Path | None:
    filename = "figure_A04_per_market_side.png"
    df = read_modeling_columns(["split", "market_id", "side_buy", TARGET])
    df = df[df["split"] == "test"].copy()
    stats = df.groupby(["market_id", "side_buy"])[TARGET].agg(hit_rate="mean", n="size").reset_index()
    if stats.empty:
        skip_figure(filename, "no test market side data")
        return None
    top_markets = df["market_id"].value_counts().head(10).index.tolist()
    pivot = stats.pivot(index="market_id", columns="side_buy", values="hit_rate").reindex(top_markets)

    figure_style()
    fig, ax = plt.subplots(figsize=(9.4, 4.6))
    x = np.arange(len(pivot))
    width = 0.36
    ax.bar(x - width / 2, pivot.get(1, pd.Series(index=pivot.index, dtype=float)), width, label="BUY", color=PAL_10[5])
    ax.bar(x + width / 2, pivot.get(0, pd.Series(index=pivot.index, dtype=float)), width, label="SELL", color=PAL_10[2])
    ax.axhline(0.5, color="0.25", ls="--", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([short_market_label(m) for m in pivot.index], rotation=35, ha="right")
    ax.set_ylabel("Bet-correct hit rate")
    ax.set_title("Per-market hit rate by trade side")
    ax.legend(frameon=False, ncols=2, loc="upper left")
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A05_single_feature_auc() -> Path | None:
    filename = "figure_A05_single_feature_auc.png"
    feature_cols = [c for c in load_feature_cols() if c in _modeling_columns()]
    df = read_modeling_columns(["split", "market_id", TARGET, *feature_cols])
    df = df[df["split"] == "test"].reset_index(drop=True)
    global_scores = []
    for feat in feature_cols:
        auc = safe_auc_score(df[TARGET].values, df[feat].values)
        if np.isfinite(auc):
            global_scores.append((feat, auc, abs(auc - 0.5)))
    if not global_scores:
        skip_figure(filename, "no valid single-feature AUCs")
        return None
    top_features = [f for f, _, _ in sorted(global_scores, key=lambda x: x[2], reverse=True)[:8]]
    top_markets = df["market_id"].value_counts().head(10).index.tolist()
    matrix = pd.DataFrame(index=top_features, columns=top_markets, dtype=float)
    for feat in top_features:
        for market in top_markets:
            sub = df[df["market_id"] == market]
            matrix.loc[feat, market] = safe_auc_score(sub[TARGET].values, sub[feat].values)

    figure_style()
    fig, ax = plt.subplots(figsize=(10.5, 5.4))
    plot_matrix = matrix.rename(index=lambda x: clean_label(x, 20), columns=short_market_label)
    sns.heatmap(
        plot_matrix.astype(float), ax=ax, annot=True, fmt=".2f",
        cmap=C_MAP_CONTRAST if "C_MAP_CONTRAST" in globals() else "vlag",
        center=0.5, vmin=0.35, vmax=0.65, cbar_kws={"label": "ROC-AUC"},
    )
    ax.set_xlabel("Test market")
    ax.set_ylabel("Feature")
    ax.set_title("Single-feature ROC-AUC by market")
    return save_fig(fig, filename)


def generate_A06_mutual_information() -> Path | None:
    filename = "figure_A06_mutual_information.png"
    cache = FIGURES_DIR / "figure_A06_mutual_information.csv"
    if cache.exists():
        mi_df = pd.read_csv(cache)
    else:
        feature_cols = [c for c in load_feature_cols() if c in _modeling_columns()]
        df = read_modeling_columns(["split", TARGET, *feature_cols])
        train = df[df["split"] == "train"].reset_index(drop=True)
        n = min(150_000, len(train))
        sample = train.sample(n=n, random_state=RANDOM_SEED)
        X = sample[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        y = sample[TARGET].astype(int).values
        print(f"  computing mutual information on {n:,} train rows × {len(feature_cols)} features ...")
        mi = mutual_info_classif(X, y, random_state=RANDOM_SEED)
        mi_df = pd.DataFrame({"feature": feature_cols, "mutual_information": mi}).sort_values(
            "mutual_information", ascending=False
        )
        mi_df.to_csv(cache, index=False)
        print(f"  cached mutual information -> {cache}")
    top = mi_df.head(25).sort_values("mutual_information")
    if top.empty:
        skip_figure(filename, "no mutual-information rows")
        return None

    figure_style()
    fig, ax = plt.subplots(figsize=(7.4, 7.0))
    colors = rocket_gradient(len(top), 0.25, 0.9) if "rocket_gradient" in globals() else None
    ax.barh([clean_label(f, 26) for f in top["feature"]], top["mutual_information"], color=colors)
    ax.set_xlabel("Mutual information")
    ax.set_title("Top 25 features by mutual information")
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A07_feature_taxonomy() -> Path | None:
    filename = "figure_A07_feature_taxonomy.png"
    path = OUTPUTS_DIR / "data" / "feature_taxonomy.json"
    if not path.exists():
        skip_figure(filename, "run Script 02 first: feature_taxonomy.json is missing")
        return None
    payload = json.loads(path.read_text())
    counts = payload.get("counts") or {k: len(v) for k, v in payload.get("groups", {}).items()}
    df = pd.Series(counts).sort_values()
    if df.empty:
        skip_figure(filename, "feature taxonomy is empty")
        return None

    figure_style()
    fig, ax = plt.subplots(figsize=(6.8, 4.0))
    ax.barh([clean_label(i, 22) for i in df.index], df.values, color=rocket_gradient(len(df), 0.25, 0.85))
    ax.set_xlabel("Feature count")
    ax.set_title("Feature counts by engineering layer")
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A08_daily_base_rate() -> Path | None:
    filename = "figure_A08_daily_base_rate.png"
    df = read_modeling_columns(["split", "ts_dt", "timestamp", TARGET])
    if "ts_dt" in df.columns:
        dates = pd.to_datetime(df["ts_dt"], utc=True, errors="coerce").dt.floor("D")
    else:
        dates = pd.to_datetime(df["timestamp"], unit="s", utc=True, errors="coerce").dt.floor("D")
    df = df.assign(date=dates).dropna(subset=["date"])
    daily = df.groupby(["date", "split"], observed=True)[TARGET].agg(hit_rate="mean", n="size").reset_index()
    all_daily = df.groupby("date")[TARGET].mean().sort_index()
    rolling = all_daily.rolling(7, min_periods=3).mean()

    figure_style()
    fig, ax = plt.subplots(figsize=(9.0, 4.2))
    colors = {"train": COL_TRAIN if "COL_TRAIN" in globals() else "0.35", "test": COL_TEST if "COL_TEST" in globals() else "0.65"}
    for split, sub in daily.groupby("split"):
        ax.scatter(sub["date"], sub["hit_rate"], s=np.clip(sub["n"] / 350, 8, 55), alpha=0.5,
                   color=colors.get(split, "0.5"), label=f"{split} daily")
    ax.plot(rolling.index, rolling.values, color="0.12", lw=2.0, label="7-day rolling mean")
    ax.axhline(0.5, color="0.25", ls="--", lw=0.8)
    ax.set_ylabel("Bet-correct base rate")
    ax.set_xlabel("Trade date")
    ax.set_title("Daily base rate with 7-day rolling mean")
    ax.legend(frameon=False, ncols=3, loc="upper left")
    fig.autofmt_xdate()
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A09_daily_volume() -> Path | None:
    filename = "figure_A09_daily_volume.png"
    ctx = read_backtest_columns(["split", "timestamp", "usd_amount"])
    ctx = ctx.assign(date=pd.to_datetime(ctx["timestamp"], unit="s", utc=True, errors="coerce").dt.floor("D"))
    daily = ctx.dropna(subset=["date"]).groupby(["date", "split"], observed=True)["usd_amount"].sum().unstack(fill_value=0).sort_index()
    if daily.empty:
        skip_figure(filename, "no daily volume rows")
        return None

    figure_style()
    fig, ax = plt.subplots(figsize=(9.0, 4.2))
    bottom = np.zeros(len(daily))
    for split, color in [("train", COL_TRAIN if "COL_TRAIN" in globals() else "0.35"),
                         ("test", COL_TEST if "COL_TEST" in globals() else "0.65")]:
        values = daily[split].values if split in daily.columns else np.zeros(len(daily))
        ax.bar(daily.index, values, bottom=bottom, width=0.9, color=color, alpha=0.85, label=split)
        bottom += values
    ax.set_ylabel("USD trade volume")
    ax.set_xlabel("Trade date")
    ax.set_title("Daily trade volume by split")
    ax.legend(frameon=False, ncols=2, loc="upper left")
    fig.autofmt_xdate()
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A10_permutation_importance() -> Path | None:
    filename = "figure_A10_permutation_importance.png"
    path = OUTPUTS_DIR / "metrics" / "permutation_importance_lightgbm.csv"
    if not path.exists():
        skip_figure(filename, "run Script 04 first: permutation_importance_lightgbm.csv is missing")
        return None
    df = pd.read_csv(path)
    if "auc_drop_mean" not in df.columns or "feature" not in df.columns:
        skip_figure(filename, "permutation importance CSV has unexpected columns")
        return None
    top = df.sort_values("auc_drop_mean", ascending=False).head(15).sort_values("auc_drop_mean")

    figure_style()
    fig, ax = plt.subplots(figsize=(7.4, 5.4))
    xerr = top["auc_drop_std"] if "auc_drop_std" in top.columns else None
    ax.barh([clean_label(f, 25) for f in top["feature"]], top["auc_drop_mean"], xerr=xerr,
            color=rocket_gradient(len(top), 0.25, 0.9) if "rocket_gradient" in globals() else None)
    ax.set_xlabel("AUC drop after permutation")
    ax.set_title("Permutation importance, LightGBM top 15")
    clean_ax(ax)
    return save_fig(fig, filename)


def tuning_models_with_history() -> list[str]:
    tuning_dir = OUTPUTS_DIR / "tuning"
    if not tuning_dir.exists():
        return []
    return [m for m in HEADLINE_TUNED_MODELS if (tuning_dir / m / "study_history.csv").exists()]


def read_study_history(model: str) -> pd.DataFrame:
    path = OUTPUTS_DIR / "tuning" / model / "study_history.csv"
    df = pd.read_csv(path)
    df["model"] = model
    df["value"] = pd.to_numeric(df.get("value"), errors="coerce")
    if "duration_sec" in df.columns:
        df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors="coerce")
    return df


def generate_A12_tuning_convergence() -> Path | None:
    filename = "figure_A12_tuning_convergence.png"
    models = tuning_models_with_history()
    if not models:
        skip_figure(filename, "no tuning artifacts found; run Script 06 first")
        return None
    histories = [read_study_history(m) for m in models]

    figure_style()
    fig, ax = plt.subplots(figsize=(7.8, 4.4))
    for df in histories:
        df = df.dropna(subset=["trial", "value"]).sort_values("trial")
        if df.empty:
            continue
        ax.plot(df["trial"], df["value"].cummax(), marker="o", ms=3, lw=1.5, label=df["model"].iloc[0])
    ax.set_xlabel("Optuna trial")
    ax.set_ylabel("Best CV AUC so far")
    ax.set_title("Tuning convergence")
    ax.legend(frameon=False, ncols=2)
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A13_default_vs_tuned_oof() -> Path | None:
    filename = "figure_A13_default_vs_tuned_oof.png"
    rows = []
    for model in tuning_models_with_history():
        metrics_path = OUTPUTS_DIR / "models" / model / "metrics.json"
        best_path = OUTPUTS_DIR / "tuning" / model / "best_params.json"
        if not metrics_path.exists() or not best_path.exists():
            print(f"  figure A.13: skipped {model}; missing metrics or best_params")
            continue
        baseline = json.loads(metrics_path.read_text()).get("cv_oof_auc")
        tuned = json.loads(best_path.read_text()).get("best_oof_auc")
        if baseline is None or tuned is None:
            print(f"  figure A.13: skipped {model}; missing OOF AUC values")
            continue
        rows.append({"model": model, "baseline_oof_auc": float(baseline), "tuned_oof_auc": float(tuned)})
    if not rows:
        skip_figure(filename, "no models have both baseline and tuned OOF AUC")
        return None
    df = pd.DataFrame(rows)

    figure_style()
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    x = np.arange(len(df))
    width = 0.36
    ax.bar(x - width / 2, df["baseline_oof_auc"], width, label="Default", color=PAL_10[3])
    ax.bar(x + width / 2, df["tuned_oof_auc"], width, label="Tuned", color=PAL_10[6])
    ax.set_ylim(max(0.5, df[["baseline_oof_auc", "tuned_oof_auc"]].min().min() - 0.02),
                min(1.0, df[["baseline_oof_auc", "tuned_oof_auc"]].max().max() + 0.03))
    ax.set_xticks(x)
    ax.set_xticklabels([clean_label(m, 16) for m in df["model"]])
    ax.set_ylabel("OOF CV ROC-AUC")
    ax.set_title("Default vs tuned OOF AUC")
    ax.legend(frameon=False, ncols=2, loc="upper left")
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A14_tuning_duration() -> Path | None:
    filename = "figure_A14_tuning_duration.png"
    rows = []
    for model in tuning_models_with_history():
        df = read_study_history(model)
        if "duration_sec" not in df.columns:
            continue
        for val in df["duration_sec"].dropna():
            if val > 0:
                rows.append({"model": model, "duration_min": float(val) / 60.0})
    if not rows:
        skip_figure(filename, "no tuning trial durations found")
        return None
    df = pd.DataFrame(rows)
    models = [m for m in HEADLINE_TUNED_MODELS if m in set(df["model"])]
    data = [df.loc[df["model"] == m, "duration_min"].values for m in models]

    figure_style()
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    bp = ax.boxplot(data, labels=[clean_label(m, 14) for m in models], patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], rocket_gradient(len(models), 0.25, 0.85)):
        patch.set_facecolor(color)
        patch.set_alpha(0.85)
    ax.set_ylabel("Trial duration (minutes)")
    ax.set_title("Optuna trial wall-clock duration")
    clean_ax(ax)
    return save_fig(fig, filename)


def generate_A15_roi_delta() -> Path | None:
    filename = "figure_A15_roi_delta.png"
    baseline_path = OUTPUTS_DIR / "_baseline_snapshot" / "backtest" / "sensitivity.csv"
    tuned_path = OUTPUTS_DIR / "backtest" / "sensitivity.csv"
    if not baseline_path.exists():
        skip_figure(filename, "missing outputs/_baseline_snapshot/backtest/sensitivity.csv; promote after a baseline Script 05 run")
        return None
    if not tuned_path.exists():
        skip_figure(filename, "missing current outputs/backtest/sensitivity.csv")
        return None
    base = pd.read_csv(baseline_path)
    tuned = pd.read_csv(tuned_path)
    keys = ["model", "strategy", "initial_capital", "max_bet_pct", "liquidity_scaler"]
    missing = [c for c in [*keys, "roi"] if c not in base.columns or c not in tuned.columns]
    if missing:
        skip_figure(filename, f"sensitivity CSVs missing columns: {missing}")
        return None
    merged = tuned.merge(base, on=keys, suffixes=("_tuned", "_baseline"))
    if merged.empty:
        skip_figure(filename, "baseline and current sensitivity tables have no matching rows")
        return None
    merged["roi_delta"] = merged["roi_tuned"] - merged["roi_baseline"]
    pivot = merged.groupby(["model", "strategy"])["roi_delta"].mean().unstack()
    if pivot.empty:
        skip_figure(filename, "ROI deltas are empty")
        return None

    figure_style()
    fig, ax = plt.subplots(figsize=(11.0, 4.8))
    vmax = float(np.nanmax(np.abs(pivot.values))) if np.isfinite(pivot.values).any() else 0.01
    sns.heatmap(
        pivot, ax=ax, cmap=C_MAP_PERFORMANCE if "C_MAP_PERFORMANCE" in globals() else "RdYlGn",
        center=0, vmin=-vmax, vmax=vmax, annot=True, fmt="+.2%",
        cbar_kws={"label": "Mean ROI delta"},
    )
    ax.set_xlabel("Strategy")
    ax.set_ylabel("Model")
    ax.set_title("Mean ROI delta after tuning")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    return save_fig(fig, filename)


def generate_main_body_figures() -> list[Path | None]:
    figure_style()
    return [
        copy_paper_figure(OUTPUTS_DIR / "metrics" / "reliability_combined.png", "figure_01_reliability.png"),
        generate_figure_02_baseline_vs_tuned(),
        copy_paper_figure(OUTPUTS_DIR / "metrics" / "per_market_per_model_auc.png", "figure_03_per_market_auc.png"),
        copy_paper_figure(OUTPUTS_DIR / "backtest" / "overview.png", "figure_04_strategy_roi_grid.png"),
    ]


def generate_eda_appendix_figures() -> list[Path | None]:
    return [
        generate_A01_train_test_shift(),
        generate_A02_time_to_deadline(),
        generate_A03_wallet_strata(),
        generate_A04_per_market_side(),
        generate_A05_single_feature_auc(),
        generate_A06_mutual_information(),
        generate_A07_feature_taxonomy(),
        generate_A08_daily_base_rate(),
        generate_A09_daily_volume(),
    ]


def generate_model_appendix_figures() -> list[Path | None]:
    figure_style()
    return [
        generate_A10_permutation_importance(),
        copy_paper_figure(OUTPUTS_DIR / "metrics" / "inter_model_agreement.png", "figure_A11_inter_model_agreement.png"),
    ]


def generate_tuning_appendix_figures() -> list[Path | None]:
    return [
        generate_A12_tuning_convergence(),
        generate_A13_default_vs_tuned_oof(),
        generate_A14_tuning_duration(),
        generate_A15_roi_delta(),
    ]


### Generate Paper Figures — Main Body (Figures 1-4)

Copies Figures 1, 3, and 4 from Script 04/05 outputs and generates Figure 2 when baseline+tuned predictions exist.


In [ ]:

generated_main = generate_main_body_figures()
print(f"Main-body figures written/skipped. Directory: {FIGURES_DIR}")


### Generate Paper Figures — EDA Appendix (A.1-A.9)

Builds the exploratory appendix figures directly from the modeling parquet and backtest context. Mutual information is cached under `outputs/figures/` after the first run.


In [ ]:

generated_eda = generate_eda_appendix_figures()
print(f"EDA appendix figures written/skipped. Directory: {FIGURES_DIR}")


### Generate Paper Figures — Model Appendix (A.10-A.11)

Creates the permutation-importance figure and copies the inter-model agreement matrix into the paper figure directory.


In [ ]:

generated_model_appendix = generate_model_appendix_figures()
print(f"Model appendix figures written/skipped. Directory: {FIGURES_DIR}")


### Generate Paper Figures — Tuning Appendix (A.12-A.15)

Generates tuning figures only when Optuna artifacts exist. Figure A.15 also requires a baseline sensitivity snapshot plus refreshed tuned backtest results.


In [ ]:

generated_tuning_appendix = generate_tuning_appendix_figures()
print(f"Tuning appendix figures written/skipped. Directory: {FIGURES_DIR}")
